# Navier–Stokes Re=500 | Resume existing runs + Boussinesq-style numerical comparison

### Reuses everything already trained. F-FNO training is disabled.

**Evaluation diagnostics:** full stdout/stderr is saved in `RECOVERY_COMPARISON_NO_FFNO_v1/evaluation_stdout.log`; per-model tracebacks are saved in `evaluation_errors.log`. Optional original-model vorticity plots are disabled by default so they cannot interrupt recovering baseline test results.

This is a **new companion notebook**, not a replacement for the previous 40-run AWFNO experiment or its 16-run extension. It uses the **same saved checkpoint names, configuration hashes, dataset, normalization, optimizer, and training protocols**. It resumes only missing **CNO**, **U-Net**, and **UNO-H10** runs at $N=25,50,100,200$. Previous completed fits, including F-FNO, are never retrained. The existing **24-layer F-FNO implementation remains in the code**, but it is **excluded from all training and held-out evaluation by default**.

| Model | Inference observations | Training method | Recovery behavior |
|---|---|---|---|
| CNO-DirectTime | 1 true initial frame | Direct-time queries | Cached or resume |
| UNet-AR1 | 1 true initial frame | One-step autoregressive | Cached or resume |
| UNO2D-H10 | 10 true initial frames | 40-step differentiable rollout | Cached or resume |
| FFNO24-AR1 | 1 true initial frame | 24-layer factorized 1-step autoregressive | **Keep files, no train/test** |

**Existing originals remain untouched:** AWFNO, TB-AWFNO, WFNO, WFNO-PM, and standard FNO. The original five-model locked publication table is included in joint comparisons *only if it exists*.

### What you see without waiting for all runs

1. A cumulative **validation table**, including previously saved F-FNO results for context (clearly excluded).
2. Held-out **70-trajectory test errors** for each *completed* CNO / U-Net / UNO fit; saved **one by one** with no re-evaluation on rerun.
3. Mean physical space–time $L^2$, final $L^2$, common $t_{10}\ldots t_{100}$ forecast $L^2$, periodic-gradient relative $H^1$, final spectral low/high-band and sharp-gradient-region diagnostics.
4. Model-by-$N$ figures, per-case error-vs-time curves, and **truth / prediction / error vorticity panels**. Combined AWFNO/baseline plots are produced when the original locked results are present.

**Scope/fairness:** UNO-H10 sees 10 exact fields. Full-horizon error is not equal-input comparable; the common *forecast-only* $t_{10}\ldots t_{100}$ result removes UNO's observed initial frames from the error integral but **does not remove its information advantage**. These are adapted reference-oriented architectures, not authors' official unchanged scripts.

**Test protocol:** Test results are released incrementally for *predeclared architectures and checkpoints*. Do **not** use test errors to tune architectures or choose models; use the validation scores for that. Saved tests are isolated under `RECOVERY_COMPARISON_NO_FFNO_v1`, avoiding changes to the previous 16-run test lock.

**Run:** select a GPU runtime; choose **Runtime → Run all**. Full execution can exceed one session; rerun this notebook to resume, without changing the settings. Do not run it simultaneously with the earlier training notebook against the same Google Drive checkpoints.


In [ ]:
#@title 1. Drive paths and recovery settings — match previous v3 run exactly
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, time, subprocess, py_compile
import numpy as np, pandas as pd, torch
from IPython.display import display, Image

assert torch.cuda.is_available(), 'Select Runtime → Change runtime type → GPU'
DATASET=Path('/content/drive/MyDrive/Kolmogorov_NS_Re500_start0.25_horizon0.25_dt0.0025_v1/kolmogorov_dataset.pt')
ORIGINAL_ROOT=Path('/content/drive/MyDrive/AWFNO_NS_Re500_101_CLEAN_FULL_v1')
NEW_ROOT=Path('/content/drive/MyDrive/AWFNO_NS_Re500_101_NATIVE_BASELINES_H10_v3')
RECOVERY_ROOT=NEW_ROOT/'RECOVERY_COMPARISON_NO_FFNO_v1'
assert DATASET.is_file(), f'Original dataset missing: {DATASET}'
NEW_ROOT.mkdir(parents=True,exist_ok=True)
RECOVERY_ROOT.mkdir(parents=True,exist_ok=True)

TRAIN_MISSING=True #@param {type:'boolean'}
EVALUATE_FINISHED=True #@param {type:'boolean'}
SESSION_MINUTES=180 #@param {type:'integer'}
PLOT_ORIGINAL_FIELDS=False #@param {type:'boolean'}
# DO NOT change training defaults: original v3 max_epochs=45, batch_size=8, seed=1234.
print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Verified dataset:',DATASET,flush=True)
print('REUSE (unchanged):',NEW_ROOT,flush=True)
print('NEW comparison outputs:',RECOVERY_ROOT,flush=True)
print('F-FNO training: DISABLED',flush=True)


## 2. Original benchmark utilities (byte-for-byte preserved)

This cell writes the existing Navier–Stokes source to `/content/ns_base_original.py`. It does **not** execute original 40-run training. Expand the cell to inspect plain Python.

In [ ]:
%%writefile /content/ns_base_original.py
#!/usr/bin/env python3
"""Independent, auditable 2D Kolmogorov Re=500 neural-operator comparison.

Do not import older Burgers/Boussinesq/Navier-Stokes model implementations.
All methods receive ONE observed initial vorticity frame; their temporal
prediction formulations are explicitly different and labelled.

Targets: 128x128 vorticity, 101 frames over physical forecast time 0..0.25.
Models: AWFNO-GLL(K20), TB-AWFNO-GLL(K20, train-only POD temporal basis),
         WFNO(K20), parameter-matched WFNO(K20), FNO(direct 100-frame).
All quantitative figures are computed with dx*dy and dt physical quadrature.
"""
from __future__ import annotations
import argparse, csv, hashlib, json, math, os, random, sys, time, shutil
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

VERSION = 'ns-re500-original-method-comparison-clean-v1'
EXPECTED_DATA_SHA = '7030bc43e6331cdf472182ae0ed4f2684c18da8d336ca9ed92c37baf0bfc8a06'
PARENT_SHA = '82786a4293ac29bf23756c2cc39188e3b7fbb84bd26ef47061415fe76103ebb3'
MODELS = ('AWFNO', 'TB-AWFNO', 'WFNO', 'WFNO-PM', 'FNO')
SIZES = (25, 50, 100, 200)
NX = 128
NT = 101
DT = 0.0025
LX = 2 * math.pi
DX = LX / NX
K_DEFAULT = 20


def emit(msg):
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)


def atomic_json(path, obj):
    path = Path(path)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    temp.replace(path)


def hash_config(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode()).hexdigest()[:14]


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def load_source(path, verify_hash=False):
    path = Path(path)
    assert path.is_file(), f'Missing dataset: {path}'
    meta_path = path.parent / 'metadata.json'
    assert meta_path.is_file(), f'Missing numerical verification metadata: {meta_path}'
    meta = json.loads(meta_path.read_text())
    assert meta['status'] == 'PASS'
    assert meta['sha256'] == EXPECTED_DATA_SHA
    assert meta['parent_sha256'] == PARENT_SHA
    assert meta['shape'] == [300, 128, 128, 101]
    if verify_hash:
        assert sha256_file(path) == EXPECTED_DATA_SHA, 'Dataset SHA256 mismatch'
    try:
        data = torch.load(path, map_location='cpu', weights_only=True, mmap=True)
    except (TypeError, RuntimeError):
        data = torch.load(path, map_location='cpu', weights_only=True)
    U = data['U']
    assert U.shape == (300, NX, NX, NT) and U.dtype == torch.float32
    assert data['parent_dataset_sha256'] == PARENT_SHA
    assert math.isclose(float(data['forecast_start']), 0.25, abs_tol=1e-7)
    assert math.isclose(float(data['T_forecast']), 0.25, abs_tol=1e-7)
    t = np.asarray(data['times'], dtype=np.float64)
    assert np.allclose(t, np.arange(NT) * DT, atol=2e-7)
    for key, lo, hi in (('train_indices', 0, 200), ('val_indices', 200, 230), ('test_indices', 230, 300)):
        assert torch.equal(data[key], torch.arange(lo, hi)), f'Split mismatch for {key}'
    x = np.asarray(data['x'], dtype=np.float64)
    y = np.asarray(data['y'], dtype=np.float64)
    assert x.shape == (NX,) and y.shape == (NX,)
    assert np.allclose(x, np.arange(NX) * DX, atol=1e-6)
    assert np.allclose(y, np.arange(NX) * DX, atol=1e-6)
    return U


def train_stats(U, indexes):
    # Training-only statistics; streaming to avoid >1 GiB temporary arrays.
    summ = sqsum = 0.0
    count = 0
    for idx in indexes:
        w = U[int(idx)].numpy()
        summ += float(np.sum(w, dtype=np.float64))
        sqsum += float(np.sum(w.astype(np.float64) ** 2))
        count += w.size
    avg = summ / count
    std = math.sqrt(max(sqsum / count - avg * avg, 1e-10))
    return float(avg), float(std)


class FramePairs(Dataset):
    """Window samples: (u(t), u(t+1),...,u(t+K)); full source trajectories split by ID."""
    def __init__(self, U, indexes, K, stride):
        self.U = U
        self.ids = [int(x) for x in indexes]
        self.K = int(K)
        self.starts = list(range(0, NT-K, stride))
        assert self.starts and self.starts[-1] + K < NT
    def __len__(self):
        return len(self.ids) * len(self.starts)
    def __getitem__(self, ix):
        i = self.ids[ix // len(self.starts)]
        s = self.starts[ix % len(self.starts)]
        v = self.U[i]
        # (1,H,W), (K,H,W); no future-state inputs at inference.
        return v[...,s].unsqueeze(0), v[...,s+1:s+1+self.K].permute(2,0,1).contiguous()


class FullTrajectories(Dataset):
    """Full-FNO training: (u0 -> 100 future fields), one sample per trajectory."""
    def __init__(self, U, indexes):
        self.U = U
        self.ids = [int(x) for x in indexes]
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, ix):
        u = self.U[self.ids[ix]]
        return u[...,0].unsqueeze(0), u[...,1:].permute(2,0,1).contiguous()


def periodic_features(device, dtype=torch.float32):
    axis = 2 * torch.pi * torch.arange(NX, device=device, dtype=dtype) / NX
    x, y = torch.meshgrid(axis, axis, indexing='ij')
    return torch.stack((x.sin(), x.cos(), y.sin(), y.cos(),
                        (4*y).sin(), (4*y).cos()), dim=0).unsqueeze(0)


class Fourier2d(nn.Module):
    """2D truncated rFFT with BOTH positive and negative x frequency blocks."""
    def __init__(self, in_ch, out_ch, modes):
        super().__init__()
        assert 1 <= modes <= NX//2
        self.modes = modes
        self.a = nn.Parameter(torch.randn(in_ch, out_ch, modes, modes, dtype=torch.cfloat)
                              / math.sqrt(in_ch*out_ch))
        self.b = nn.Parameter(torch.randn(in_ch, out_ch, modes, modes, dtype=torch.cfloat)
                              / math.sqrt(in_ch*out_ch))
    def forward(self, x):
        b, _, h, w = x.shape
        m = self.modes
        assert 2*m <= h and m <= w//2 + 1
        X = torch.fft.rfft2(x, norm='ortho')
        Y = torch.zeros((b, self.a.shape[1],h,w//2+1),device=x.device,dtype=X.dtype)
        Y[:,:,:m,:m] = torch.einsum('bixy,ioxy->boxy', X[:,:,:m,:m],self.a)
        Y[:,:,-m:,:m] = torch.einsum('bixy,ioxy->boxy', X[:,:,-m:,:m],self.b)
        return torch.fft.irfft2(Y,s=(h,w),norm='ortho')


class MixedBlock(nn.Module):
    def __init__(self, width, modes, radius=2, order='GLL'):
        super().__init__()
        assert order in ('LGL','GLL','LLG','GGG','LLL')
        self.ops=nn.ModuleList([
            Fourier2d(width,width,modes) if x=='G' else nn.Conv2d(
                width,width,2*radius+1,padding=radius,padding_mode='circular')
            for x in order
        ])
    def forward(self,x):
        for op in self.ops:
            x = x + F.gelu(op(x))
        return x


class FourierBlock(nn.Module):
    """Windowed FNO block: one spectral map + pointwise path per depth."""
    def __init__(self,width,modes):
        super().__init__()
        self.spectral=Fourier2d(width,width,modes)
        self.pointwise=nn.Conv2d(width,width,1)
    def forward(self,x):
        return F.gelu(self.spectral(x)+self.pointwise(x))


class OperatorModel(nn.Module):
    """All variants have exactly ONE initial observed field at test time."""
    def __init__(self, method, width, modes, depth, K, rank=12, order='GLL', radius=2):
        super().__init__()
        assert method in MODELS
        self.method, self.width, self.K, self.rank = method, width, K, rank
        self.register_buffer('grid',periodic_features('cpu'),persistent=False)
        self.lift=nn.Conv2d(1+6,width,1)
        mixed=method in ('AWFNO','TB-AWFNO')
        self.blocks=nn.ModuleList([
            MixedBlock(width,modes,radius,order) if mixed else FourierBlock(width,modes)
            for _ in range(depth)
        ])
        self.hidden=nn.Conv2d(width,128,1)
        outc=rank if method=='TB-AWFNO' else (NT-1 if method=='FNO' else K)
        self.final=nn.Conv2d(128,outc,1)
        if method=='TB-AWFNO':
            assert 1 <= rank <= K
            self.register_buffer('temporal_basis',torch.zeros(K,rank))
    def forward(self, u0):
        assert u0.ndim == 4 and u0.shape[1] == 1
        x=torch.cat((u0,self.grid.to(device=u0.device,dtype=u0.dtype).expand(u0.shape[0],-1,-1,-1)),dim=1)
        x=self.lift(x)
        for layer in self.blocks:
            x=layer(x)
        z=self.final(F.gelu(self.hidden(x)))
        if self.method=='TB-AWFNO':
            # basis[k,r] fit on TRAIN trajectories only. Predict *increments*.
            increment=torch.einsum('brxy,kr->bkxy',z,self.temporal_basis)
            return u0+increment
        return z


def real_dof(model):
    return int(sum(p.numel()*(2 if p.is_complex() else 1) for p in model.parameters() if p.requires_grad))


def param_count(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def match_wfno_width(target, modes, depth, K, tolerance=.05):
    candidates=[]
    for width in range(12,97):
        m=OperatorModel('WFNO-PM',width,modes,depth,K)
        n=real_dof(m)
        candidates.append((abs(n-target)/target,width,n))
        del m
    best=min(candidates)
    if best[0]>tolerance:
        raise RuntimeError(f'Unable to match real parameter count within {tolerance:.0%}: {best}')
    return best[1],best[2],best[0]


def compute_train_basis(U, train_ids, K, rank, stride=10, spatial_stride=4):
    """Training-only POD for residual windows, with deterministic uniform spatial quadrature.

    R has shape (sampled space-time locations, K); its covariance R^T R
    yields orthonormal temporal modes phi_j(t) for TB-AWFNO reconstruction.
    """
    cov=np.zeros((K,K),dtype=np.float64)
    rows=0
    for i in train_ids:
        w=U[int(i)].numpy()[::spatial_stride,::spatial_stride,:]
        for start in range(0,NT-K,stride):
            delta=w[...,start+1:start+1+K]-w[...,start,None]
            X=delta.reshape(-1,K).astype(np.float64,copy=False)
            cov+=X.T@X
            rows+=len(X)
    vals,vecs=np.linalg.eigh(cov)
    take=np.argsort(vals)[::-1]
    vals=np.maximum(vals[take],0)
    basis=vecs[:,take[:rank]].astype(np.float32)
    # Deterministic sign: largest magnitude component is positive.
    for col in range(rank):
        v=basis[:,col]
        if v[np.argmax(np.abs(v))]<0:basis[:,col]*=-1
    energy=float(vals[:rank].sum()/max(vals.sum(),1e-30))
    assert np.allclose(basis.T@basis,np.eye(rank),atol=3e-5)
    return torch.from_numpy(basis.copy()),energy,rows


def make_model(method, args, pm_width):
    width=pm_width if method=='WFNO-PM' else args.width
    return OperatorModel(method,width,args.modes,args.depth,args.window_k,
                         args.basis_rank,args.order,args.radius)


def load_normalized_stats(U, ids, output):
    # Reuse TRAIN-only normalization across the five methods at each N.
    path=output/f'train_normalization_N{len(ids)}.json'
    if path.is_file():
        ob=json.loads(path.read_text())
        assert ob['dataset_sha256']==EXPECTED_DATA_SHA and ob['ntrain']==len(ids)
        return float(ob['mean']),float(ob['std'])
    mean,std=train_stats(U,ids)
    assert std>0 and np.isfinite(std)
    atomic_json(path,{'dataset_sha256':EXPECTED_DATA_SHA,'ntrain':len(ids),
                      'mean':mean,'std':std,'fit_split':'train_only'})
    return mean,std


def prepare_batch(batch, device, mean, std):
    inp, future=batch
    return ((inp.to(device,non_blocking=True)-mean)/std,
            (future.to(device,non_blocking=True)-mean)/std)


def model_predictions(model, U, ids, device, mean, std, batch_size=3):
    """Full *free* rollouts. No true post-t0 observations are passed into models."""
    model.eval()
    predicted=[]; truth=[]
    with torch.inference_mode():
        for ix in range(0,len(ids),batch_size):
            selection=list(ids[ix:ix+batch_size])
            source=torch.stack([U[int(i)] for i in selection],dim=0)
            initial=((source[...,0].unsqueeze(1).to(device)-mean)/std)
            if model.method=='FNO':
                frames=model(initial)
                assert frames.shape[1]==NT-1
                parts=[frames]
            else:
                current=initial
                parts=[]
                for start in range(0,NT-1,model.K):
                    y=model(current)
                    length=min(model.K,NT-1-start)
                    parts.append(y[:,:length])
                    current=y[:,length-1:length].detach()
            output=torch.cat(parts,dim=1)
            assert output.shape==(len(selection),NT-1,NX,NX)
            full=torch.cat((initial,output),dim=1)*std+mean
            # (B,T,H,W) for both truth and prediction
            predicted.append(full.float().cpu().numpy())
            truth.append(source.permute(0,3,1,2).float().numpy())
    return np.concatenate(predicted,axis=0),np.concatenate(truth,axis=0)


def l2_metrics(pred,truth):
    """Absolute physical L2(Ω×T) with trapezoidal time quadrature; mean per trajectory."""
    assert pred.shape==truth.shape and pred.ndim==4
    error=(pred.astype(np.float64)-truth.astype(np.float64))
    l2=np.sqrt(DX*DX*np.sum(error**2,axis=(2,3)))
    weights=np.ones(NT,dtype=np.float64)
    weights[0]=weights[-1]=0.5
    full=np.sqrt(DT*np.sum((l2**2)*weights[None,:],axis=1))
    final=l2[:,-1]
    return {'full':full,'final':final,'per_timestep':l2}


def moment(xs):
    a=np.asarray(xs,dtype=np.float64)
    return dict(mean=float(a.mean()),std=float(a.std(ddof=0)),median=float(np.median(a)))


def spectral_region_metrics(pred,truth):
    """Final-time low/high Fourier error and high-gradient-region relative error."""
    final_p=pred[:,-1].astype(np.float64)
    final_t=truth[:,-1].astype(np.float64)
    k=np.fft.fftfreq(NX)*NX
    band=np.maximum(np.abs(k[:,None]),np.abs(k[None,:]))
    E=np.fft.fft2(final_p-final_t,axes=(-2,-1),norm='ortho')
    U=np.fft.fft2(final_t,axes=(-2,-1),norm='ortho')
    results={}
    for name,mask in (('low',band<16),('high',(band>=16)&(band<43))):
        en=np.sum(np.abs(E[:,mask])**2,axis=1)
        tn=np.sum(np.abs(U[:,mask])**2,axis=1)
        results[f'{name}_frequency_absolute_l2']=moment(np.sqrt(DX*DX*en))
        results[f'{name}_frequency_relative_l2']=moment(np.sqrt(en/np.maximum(tn,1e-20)))
    gx=(np.roll(final_t,-1,axis=1)-np.roll(final_t,1,axis=1))/(2*DX)
    gy=(np.roll(final_t,-1,axis=2)-np.roll(final_t,1,axis=2))/(2*DX)
    grad=np.hypot(gx,gy)
    mask=grad>=np.quantile(grad,0.9,axis=(1,2),keepdims=True)
    e2=(final_p-final_t)**2
    rel=np.sqrt(np.sum(e2*mask,axis=(1,2)) /
                np.maximum(np.sum(final_t**2*mask,axis=(1,2)),1e-30))
    results['sharp_region_relative_l2']=moment(rel)
    return results


def plot_case(output,pred,truth,case,name,frame_ids=(0,25,50,75,100)):
    import matplotlib.pyplot as plt
    fig,axs=plt.subplots(3,len(frame_ids),figsize=(17,9),constrained_layout=True)
    gt=truth[case];pr=pred[case]
    vmax=float(np.max(np.abs(gt)))+1e-10
    emax=max(float(np.percentile(np.abs(pr-gt),99)),1e-6)
    for j,t in enumerate(frame_ids):
        axs[0,j].imshow(gt[t].T,origin='lower',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
        axs[1,j].imshow(pr[t].T,origin='lower',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
        axs[2,j].imshow(np.abs(pr[t]-gt[t]).T,origin='lower',cmap='magma',vmin=0,vmax=emax)
        axs[0,j].set_title(f't={t*DT:.4f}')
        for row in range(3):
            axs[row,j].set_xticks([]);axs[row,j].set_yticks([])
    for i,caption in enumerate(('True ω','Predicted ω','|Error|')):
        axs[i,0].set_ylabel(caption)
    fig.suptitle(name)
    fig.savefig(output,dpi=120)
    plt.close(fig)


def plot_training(folder,history):
    import matplotlib.pyplot as plt
    if not history:return
    f,ax=plt.subplots(figsize=(9,4),constrained_layout=True)
    epochs=[h['epoch'] for h in history]
    ax.plot(epochs,[h['train_mse'] for h in history],label='train window MSE')
    vals=[h for h in history if np.isfinite(h['val_full_l2'])]
    if vals:
        ax2=ax.twinx()
        ax2.plot([h['epoch'] for h in vals], [h['val_full_l2'] for h in vals],'-o',color='C1',label='val full L2')
        ax2.set_ylabel('Absolute validation full L2')
    ax.set_xlabel('Epoch');ax.set_ylabel('Mean normalized train-window MSE')
    ax.grid(alpha=.2)
    f.savefig(folder/'training_history.png',dpi=130)
    plt.close(f)


def plot_error_curve(folder,metrics,comparison='test'):
    import matplotlib.pyplot as plt
    per=metrics['per_timestep']
    mean=per.mean(0);std=per.std(0)
    t=np.arange(NT)*DT
    f,ax=plt.subplots(figsize=(10,4),constrained_layout=True)
    ax.plot(t,mean,label='Model',lw=2)
    ax.fill_between(t,np.maximum(0,mean-std),mean+std,alpha=.2,label='±1 std')
    ax.set(title=f'{comparison}: free-rollout absolute physical L2',xlabel='Forecast time',ylabel='L2(Ω)')
    ax.grid(alpha=.2);ax.legend()
    f.savefig(folder/f'{comparison}_error_vs_time.png',dpi=140)
    plt.close(f)
    np.savez_compressed(folder/f'{comparison}_curves.npz',t=t,mean=mean,std=std)


def elapsed_s(device,start):
    if device.type=='cuda':torch.cuda.synchronize(device)
    return time.perf_counter()-start


def run_epoch(model,loader,device,mean,std,optimizer,seed,epoch):
    model.train(); total=0.; cnt=0
    # A deterministic epoch shuffle is restored exactly on resume.
    generator=torch.Generator().manual_seed(seed+1000003*epoch)
    order=torch.randperm(len(loader.dataset),generator=generator).tolist()
    for beginning in range(0,len(order),loader.batch_size):
        idxs=order[beginning:beginning+loader.batch_size]
        batch=[loader.dataset[j] for j in idxs]
        x=torch.stack([b[0] for b in batch]);y=torch.stack([b[1] for b in batch])
        xn,yn=prepare_batch((x,y),device,mean,std)
        optimizer.zero_grad(set_to_none=True)
        predicted=model(xn)
        assert predicted.shape==yn.shape, f'Target mismatch {predicted.shape} != {yn.shape}'
        loss=F.mse_loss(predicted,yn)
        if not bool(torch.isfinite(loss)):raise FloatingPointError('Nonfinite training loss')
        loss.backward()
        grad=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
        if not bool(torch.isfinite(grad)):raise FloatingPointError('Nonfinite gradient')
        optimizer.step()
        total+=float(loss.detach().item())*len(idxs)
        cnt+=len(idxs)
    return total/cnt,math.ceil(len(order)/loader.batch_size)


def record_config(args,model,n,method,pm_width,pm_gap):
    return dict(version=VERSION,dataset_sha256=EXPECTED_DATA_SHA,
                n_train=n,seed=args.seed,method=method,epochs=args.max_epochs,
                early_stop_patience=args.patience,early_stop_min_epochs=args.min_epochs,
                val_every=args.val_every,window_stride=args.window_stride,
                k=args.window_k,basis_rank=args.basis_rank,
                order=args.order,depth=args.depth,modes=args.modes,radius=args.radius,
                width=model.width,reference_width=args.width,
                batch_size=args.batch_size,eval_batch_size=args.eval_batch,
                lr=args.lr,weight_decay=args.wd,pm_width=pm_width,
                pm_real_dof_relative_gap=pm_gap,
                normalized_targets=True,train_only_normalization=True,
                selection='best_validation_physical_absolute_L2_Omega_times_T',
                test_sealed_until_all_models_trained=True)


def model_folder(root,config):
    return root/'runs'/config['method']/f"N{config['n_train']}_seed{config['seed']}_{hash_config(config)}"


def save_checkpoint(folder,model,opt,config,epoch,best,best_epoch,best_state,history,
                    mean,std,basis_energy,optimizer_steps,training_seconds):
    tmp=folder/'checkpoint.tmp'
    payload=dict(epoch=epoch,state=model.state_dict(),optimizer=opt.state_dict(),
                 best=best,best_epoch=best_epoch,best_state=best_state,
                 history=history,config=config,mean=mean,std=std,basis_energy=basis_energy,
                 steps=optimizer_steps,elapsed=training_seconds,
                 rng_torch=torch.get_rng_state(),rng_numpy=np.random.get_state(),
                 rng_python=random.getstate(),
                 rng_cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None)
    torch.save(payload,tmp)
    tmp.replace(folder/'checkpoint.pt')


def fit_one(U,args,n,method,pm_width,pm_gap,device):
    # Avoid loading any test tensors until ALL models have been trained and selected.
    ids=np.arange(n)
    valid=np.arange(200,230)
    model=make_model(method,args,pm_width).to(device)
    config=record_config(args,model,n,method,pm_width,pm_gap)
    folder=model_folder(args.output,config);folder.mkdir(parents=True,exist_ok=True)
    done=folder/'training_result.json'
    if done.exists() and not args.force:
        emit(f'[CACHED] {method} N={n} from {done}')
        return json.loads(done.read_text())
    avg,std=load_normalized_stats(U,ids,args.output)
    if method=='TB-AWFNO':
        basis,energy,rows=compute_train_basis(U,ids,args.window_k,args.basis_rank,args.window_stride)
        model.temporal_basis.copy_(basis.to(device))
        emit(f'[TB BASIS] N={n} rank={args.basis_rank} captured energy={100*energy:.4f}% from {rows:,} training rows')
    else:energy=float('nan')
    dataset=(FullTrajectories(U,ids) if method=='FNO'
             else FramePairs(U,ids,args.window_k,args.window_stride))
    # Use Dataset's own deterministic epoch sampling for exact resumability.
    loader=DataLoader(dataset,batch_size=args.batch_size)
    optimizer=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=args.wd)
    # Fixed LR for unambiguous epoch-resume, early stopping via validation.
    history=[];best=float('inf');best_epoch=0;best_state=None;last_epoch=0
    steps=0;prev_elapsed=0.;bad=0
    ckpt=folder/'checkpoint.pt'
    if ckpt.exists() and not args.force:
        state=torch.load(ckpt,map_location='cpu',weights_only=False)
        if state['config']!=config:raise ValueError('Checkpoint config mismatch')
        model.load_state_dict(state['state'])
        optimizer.load_state_dict(state['optimizer'])
        best=state['best'];best_epoch=state['best_epoch'];best_state=state['best_state']
        history=state['history'];last_epoch=state['epoch'];steps=state['steps']
        prev_elapsed=state['elapsed']
        torch.set_rng_state(state['rng_torch'])
        np.random.set_state(state['rng_numpy']);random.setstate(state['rng_python'])
        if state['rng_cuda'] is not None and device.type=='cuda':
            torch.cuda.set_rng_state_all(state['rng_cuda'])
        bad=0
        for h in reversed(history):
            if np.isfinite(h['val_full_l2']):
                if h['epoch'] != best_epoch:bad+=1
                else:break
        emit(f'[RESUME] {method} N={n} epoch {last_epoch}/{args.max_epochs}')
    emit(f'[TRAIN] {method} N={n}: {len(dataset)} samples/epoch, '
         f'{param_count(model):,} PyTorch params / {real_dof(model):,} real DoF, '
         f'epochs <= {args.max_epochs}, training batch={args.batch_size}')
    start=time.perf_counter()
    if last_epoch==args.max_epochs and best_state is None:
        raise RuntimeError('Reached training cap but checkpoint has no best validation state')
    for epoch in range(last_epoch+1,args.max_epochs+1):
        mse,nsteps=run_epoch(model,loader,device,avg,std,optimizer,args.seed,epoch)
        steps+=nsteps
        val_full=float('nan');val_final=float('nan')
        if epoch==1 or epoch%args.val_every==0 or epoch==args.max_epochs:
            emit(f'[VALIDATE] {method} N={n} epoch {epoch}: 30 trajectories x 101 frames')
            predicted,true=model_predictions(model,U,valid,device,avg,std,args.eval_batch)
            vm=l2_metrics(predicted,true)
            val_full=moment(vm['full'])['mean'];val_final=moment(vm['final'])['mean']
            if val_full<best-args.min_delta:
                best=val_full;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                bad=0
            else:bad+=1
            del predicted,true
        history.append(dict(epoch=epoch,train_mse=mse,val_full_l2=val_full,
                            val_final_l2=val_final,optimizer_steps=steps))
        duration=prev_elapsed+elapsed_s(device,start)
        emit(f'[EPOCH] {method} N={n} epoch {epoch}/{args.max_epochs}, '
             f'loss={mse:.5e}, val_full={val_full:.6f}, '
             f'best={best:.6f}@{best_epoch}, '
             f'optimizer_steps={steps}, elapsed={duration:.1f}s')
        if epoch%args.checkpoint_every==0 or epoch==args.max_epochs or (
                epoch>=args.min_epochs and bad>=args.patience):
            save_checkpoint(folder,model,optimizer,config,epoch,best,best_epoch,best_state,
                            history,avg,std,energy,steps,duration)
        if (epoch>=args.min_epochs and bad>=args.patience):
            emit(f'[EARLY STOP] {method} N={n} after {bad} non-improving validation checks')
            break
        if time.perf_counter()-args.session_start>args.session_minutes*60:
            # Save current epoch even if not a checkpoint multiple.
            save_checkpoint(folder,model,optimizer,config,epoch,best,best_epoch,best_state,
                            history,avg,std,energy,steps,duration)
            emit('[BUDGET] Session budget reached. Rerun training to resume automatically.')
            raise SessionBudgetExceeded
    if best_state is None:raise RuntimeError('No validation checkpoint')
    model.load_state_dict(best_state)
    chosen_pred,chosen_true=model_predictions(model,U,valid,device,avg,std,args.eval_batch)
    chosen_metrics=l2_metrics(chosen_pred,chosen_true)
    plot_training(folder,history)
    plot_error_curve(folder,chosen_metrics,'validation')
    plot_case(folder/'validation_vorticity_0.png',chosen_pred,chosen_true,0,f'{method} N{n} VALIDATION example 0')
    plot_case(folder/'validation_vorticity_15.png',chosen_pred,chosen_true,15,f'{method} N{n} VALIDATION example 15')
    torch.save({'state_dict':best_state,'config':config,'mean':avg,'std':std,
                'basis_energy':energy},folder/'best_validation.pt')
    duration=prev_elapsed+elapsed_s(device,start)
    row=dict(config_hash=hash_config(config),version=VERSION,method=method,N_train=n,
             seed=args.seed,parameters=param_count(model),real_scalar_dof=real_dof(model),
             width=model.width,K=(0 if method=='FNO' else args.window_k),
             observed_input_frames=1,prediction_protocol='direct_100_frame' if method=='FNO'
             else 'windowed_basis_increment' if method=='TB-AWFNO' else 'windowed_K20',
             train_samples_per_epoch=len(dataset),optimizer_steps=steps,
             epochs_completed=history[-1]['epoch'],best_epoch=best_epoch,
             best_validation_full_l2=float(moment(chosen_metrics['full'])['mean']),
             best_validation_final_l2=float(moment(chosen_metrics['final'])['mean']),
             train_seconds=float(duration),basis_energy_fraction=None if method!='TB-AWFNO' else energy,
             pm_real_dof_relative_gap=None if method!='WFNO-PM' else pm_gap,
             checkpoint=str(folder/'best_validation.pt'))
    atomic_json(done,row)
    del model,optimizer,chosen_pred,chosen_true
    if device.type=='cuda':torch.cuda.empty_cache()
    return row


class SessionBudgetExceeded(Exception):
    pass


def all_expected(args):
    return [(n,m) for n in args.sizes for m in args.methods]


def expected_configs(args,pm_width,pm_gap):
    result=[]
    for n,method in all_expected(args):
        m=make_model(method,args,pm_width)
        cfg=record_config(args,m,n,method,pm_width,pm_gap)
        result.append(dict(method=method,N_train=n,hash=hash_config(cfg),path=str(model_folder(args.output,cfg))))
    return result


def summarize_tables(root,rows,from_test=False):
    import pandas as pd
    if not rows:return
    df=pd.DataFrame(rows).sort_values(['N_train','method']).reset_index(drop=True)
    df.to_csv(root/('results.csv' if from_test else 'training_selection.csv'),index=False)
    df.to_json(root/('results.json' if from_test else 'training_selection.json'),orient='records',indent=2)
    if from_test:
        columns=['N_train','method','K','prediction_protocol','observed_input_frames',
                 'parameters','real_scalar_dof','full_l2_mean','full_l2_std','full_l2_median',
                 'final_l2_mean','final_l2_std','final_l2_median',
                 'train_seconds','inference_seconds_70','inference_seconds_per_trajectory',
                 'peak_inference_gpu_mib','best_epoch','optimizer_steps']
        df[[c for c in columns if c in df.columns]].to_csv(root/'publication_table.csv',index=False)
        print('\n=== Boussinesq/Burgers-style model-by-training-size table (TEST) ===',flush=True)
        print(df[[c for c in columns if c in df.columns]].to_string(index=False),flush=True)
    else:
        columns=['N_train','method','parameters','real_scalar_dof',
                 'best_validation_full_l2','best_validation_final_l2','train_seconds',
                 'epochs_completed','optimizer_steps']
        print('\n=== TRAIN/VALIDATION ONLY (TEST SEALED) ===',flush=True)
        print(df[[c for c in columns if c in df.columns]].to_string(index=False),flush=True)


def plot_size_summary(root,rows):
    import matplotlib.pyplot as plt
    if not rows:return
    by={(int(r['N_train']),r['method']):r for r in rows}
    for name in ('full','final'):
        f,ax=plt.subplots(figsize=(9,5),constrained_layout=True)
        for method in MODELS:
            nlist=sorted({n for n,m in by if m==method})
            if not nlist:continue
            values=[by[(n,method)][f'{name}_l2_mean'] for n in nlist]
            ax.plot(nlist,values,'-o',label=method)
        ax.set_xlabel('Number of training trajectories')
        ax.set_ylabel(f'Mean test absolute physical {name} L2')
        ax.set_title(f'Test {name}-trajectory L2 vs training-set size')
        ax.grid(alpha=.2);ax.legend()
        f.savefig(root/f'all_models_{name}_l2_vs_n.png',dpi=140)
        plt.close(f)


def plot_cross_model_figures(root, rows):
    """Burgers/Boussinesq-format plots from completed TEST metrics, no retraining."""
    import matplotlib.pyplot as plt
    lookup={(int(r['N_train']),r['method']):r for r in rows}
    for metric,title,name in (
        ('train_seconds','Training wall time','all_models_training_time_vs_n.png'),
        ('inference_seconds_70','Full 70-case test inference time','all_models_inference_time_vs_n.png'),
        ('sharp_region_relative_l2_mean','Sharp-gradient-region error at final time',
         'all_models_sharp_region_error_vs_n.png'),
        ('high_frequency_relative_l2_mean','High-frequency relative error at final time',
         'all_models_high_frequency_error_vs_n.png')):
        fig,ax=plt.subplots(figsize=(9,5),constrained_layout=True)
        for model in MODELS:
            nlist=sorted(n for n in SIZES if (n,model) in lookup)
            if nlist:
                ax.plot(nlist,[lookup[(n,model)][metric] for n in nlist],'-o',label=model)
        ax.set(xlabel='Training trajectories',ylabel=title,title=title+' vs N')
        ax.grid(alpha=.2);ax.legend()
        fig.savefig(root/name,dpi=140)
        plt.close(fig)
    for n in SIZES:
        fig,ax=plt.subplots(figsize=(9,5),constrained_layout=True)
        count=0
        for model in MODELS:
            if (n,model) not in lookup:continue
            row=lookup[(n,model)]
            folder=Path(row['checkpoint']).parent
            p=folder/'test_curves.npz'
            if not p.is_file():continue
            arr=np.load(p)
            ax.plot(arr['t'],arr['mean'],label=model,lw=1.8)
            count+=1
        ax.set(title=f'Test mean physical L2 vs time | N={n} | {count} methods',
               xlabel='Forecast time',ylabel='Absolute physical L2(Ω)')
        ax.grid(alpha=.2);ax.legend()
        fig.savefig(root/f'all_models_error_vs_time_N{n}.png',dpi=150)
        plt.close(fig)
    if all((n,'AWFNO') in lookup and (n,'WFNO-PM') in lookup for n in SIZES):
        fig,ax=plt.subplots(figsize=(9,5),constrained_layout=True)
        for method in ('AWFNO','WFNO-PM'):
            vals=[lookup[(n,method)]['full_l2_mean'] for n in SIZES]
            ax.plot(SIZES,vals,'-o',label=method)
        ax.set(xlabel='Training trajectories',ylabel='Mean full L2(Ω×T)',
               title='Parameter-matched AWFNO versus WFNO')
        ax.grid(alpha=.2);ax.legend()
        fig.savefig(root/'matched_WFNO_vs_AWFNO_full_l2.png',dpi=150)
        plt.close(fig)


def evaluate_test(U,args,expected,device):
    assert all((Path(j['path'])/'training_result.json').is_file() for j in expected),(
        'Refusing test evaluation until ALL predeclared training runs complete')
    locked=expected
    lock_path=args.output/'LOCKED_TEST_PLAN.json'
    if lock_path.exists():
        if json.loads(lock_path.read_text())!=locked:
            raise RuntimeError('Test plan was already locked to different configurations')
    else:atomic_json(lock_path,locked)
    test_ids=np.arange(230,300)
    test_rows=[]
    for j,record in enumerate(locked,1):
        folder=Path(record['path'])
        finish=folder/'test_result.json'
        if finish.exists() and not args.force_test:
            emit(f'[CACHED TEST] {j}/{len(locked)}: {record["method"]} N{record["N_train"]}')
            test_rows.append(json.loads(finish.read_text()));continue
        training=json.loads((folder/'training_result.json').read_text())
        weight=torch.load(folder/'best_validation.pt',map_location='cpu',weights_only=False)
        config=weight['config']
        assert hash_config(config)==record['hash']
        m=OperatorModel(config['method'],config['width'],config['modes'],config['depth'],
                        config['k'],config['basis_rank'],config['order'],config['radius']).to(device)
        m.load_state_dict(weight['state_dict'])
        m.eval()
        emit(f'[TEST] {j}/{len(locked)}: {record["method"]} N={record["N_train"]} '
             f'(one initial observation, 70 trajectories)')
        if device.type=='cuda':
            torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
        started=time.perf_counter()
        pred,true=model_predictions(m,U,test_ids,device,weight['mean'],weight['std'],args.eval_batch)
        sec=elapsed_s(device,started)
        peak=float(torch.cuda.max_memory_allocated(device)/2**20) if device.type=='cuda' else 0.0
        metric=l2_metrics(pred,true)
        diagnostic=spectral_region_metrics(pred,true)
        row=dict(training,
                 full_l2_mean=moment(metric['full'])['mean'],
                 full_l2_std=moment(metric['full'])['std'],
                 full_l2_median=moment(metric['full'])['median'],
                 final_l2_mean=moment(metric['final'])['mean'],
                 final_l2_std=moment(metric['final'])['std'],
                 final_l2_median=moment(metric['final'])['median'],
                 inference_seconds_70=sec,
                 inference_seconds_per_trajectory=sec/len(test_ids),
                 peak_inference_gpu_mib=peak,
                 **{k+'_'+z:vv for k,v in diagnostic.items() for z,vv in v.items()})
        plot_error_curve(folder,metric,'test')
        plot_case(folder/'test_vorticity_0.png',pred,true,0,f'{record["method"]} N{record["N_train"]} TEST case 230')
        plot_case(folder/'test_vorticity_35.png',pred,true,35,f'{record["method"]} N{record["N_train"]} TEST case 265')
        atomic_json(finish,row)
        test_rows.append(row)
        summarize_tables(args.output,test_rows,from_test=True)
        plot_size_summary(args.output,test_rows)
        plot_cross_model_figures(args.output,test_rows)
        emit(f'[TEST COMPLETE] {j}/{len(locked)} {record["method"]} N{record["N_train"]} '
             f'full={row["full_l2_mean"]:.5f}, final={row["final_l2_mean"]:.5f}, '
             f'inference_70={sec:.2f}s')
        del pred,true,m
        if device.type=='cuda':torch.cuda.empty_cache()
    return test_rows


def smoke_test(args,device):
    """CPU/GPU test of every method, gradients, temporal basis and quadrature."""
    seed_all(args.seed)
    target=OperatorModel('AWFNO',args.width,args.modes,args.depth,args.window_k,
                         args.basis_rank,args.order,args.radius)
    ref_real=real_dof(target)
    pm_width,pm_n,gap=match_wfno_width(ref_real,args.modes,args.depth,args.window_k)
    emit(f'[PARAMETER MATCH] AWFNO real DOF={ref_real:,}; '
         f'WFNO-PM width={pm_width} real DOF={pm_n:,}; relative gap={100*gap:.2f}%')
    del target
    X=torch.randn(2,1,NX,NX,device=device)
    target_seq=torch.randn(2,args.window_k,NX,NX,device=device)
    basis=torch.linalg.qr(torch.randn(args.window_k,args.basis_rank,device=device),mode='reduced').Q
    for method in MODELS:
        model=make_model(method,args,pm_width).to(device)
        if method=='TB-AWFNO':model.temporal_basis.copy_(basis)
        target=target_seq if method!='FNO' else torch.randn(2,NT-1,NX,NX,device=device)
        out=model(X)
        assert out.shape==target.shape and bool(torch.isfinite(out).all()),method
        loss=F.mse_loss(out,target)
        loss.backward()
        assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        emit(f'[SMOKE] {method} | shape={tuple(out.shape)} | '
             f'params={param_count(model):,} | real DoF={real_dof(model):,} | gradients PASS')
        del model,out,loss
        if device.type=='cuda':torch.cuda.empty_cache()
    # Independently test the controlled component/order ablation kernels too.
    for order in ('LGL','LLG','GGG','LLL'):
        ablation=OperatorModel('AWFNO',args.width,args.modes,args.depth,
                              args.window_k,args.basis_rank,order,args.radius).to(device)
        out=ablation(X[:1])
        assert out.shape==(1,args.window_k,NX,NX) and bool(torch.isfinite(out).all()),order
        out.mean().backward()
        assert any(p.grad is not None for p in ablation.parameters())
        emit(f'[SMOKE ABLATION] {order} | params={param_count(ablation):,} | gradients PASS')
        del ablation,out
        if device.type=='cuda':torch.cuda.empty_cache()
    clean=np.ones((1,NT,NX,NX),dtype=np.float32)
    m=l2_metrics(clean,clean)
    assert m['full'].max()==0. and m['final'].max()==0.
    altered=clean.copy();altered[:,1:]+=1
    q=l2_metrics(altered,clean)
    expected=math.sqrt(LX*LX*(NT-1.5)*DT)
    assert np.allclose(q['full'],expected,rtol=1e-6)
    assert np.allclose(q['final'],LX,rtol=1e-6)
    emit('[METRIC] Physical dxdy/dt quadrature analytical tests PASS')
    return pm_width,gap


def supplementary_ablation_plan(args):
    """Fresh controlled ablations from THIS clean implementation (not the older 22)."""
    from copy import copy
    base=dict(method='AWFNO',ntrain=100,order='GLL',window_k=20,
              modes=16,depth=4,radius=2,basis_rank=12,seed=args.seed)
    jobs=[]
    def add(family,**changes):
        cfg={**base,**changes}
        cfg['family']=family
        jobs.append(cfg)
    for order in ('LGL','LLG','GGG','LLL'):
        add('ordering_and_components',order=order)
    for K in (2,5,10):
        add('temporal_window_K',window_k=K,basis_rank=min(12,K))
    for modes in (8,24,32):
        add('Fourier_modes',modes=modes)
    for radius in (1,3):
        add('local_radius',radius=radius)
    for depth in (2,6):
        add('network_depth',depth=depth)
    for rank in (8,16):
        add('temporal_basis_rank',method='TB-AWFNO',basis_rank=rank)
    for seed in (2026,7727):
        for order in ('GLL','LGL'):
            add('repeat_seed',order=order,seed=seed)
    assert len(jobs)==20
    assert len({tuple(j[x] for x in ('method','ntrain','order','window_k',
                        'modes','depth','radius','basis_rank','seed')) for j in jobs})==len(jobs)
    return jobs


def do_supplementary_ablations(U,args,pm_width,pm_gap,device):
    from copy import copy
    output=args.output/'independent_ablations'
    output.mkdir(parents=True,exist_ok=True)
    jobs=supplementary_ablation_plan(args)
    manifest=output/'predeclared_controlled_studies.json'
    if manifest.exists():
        if json.loads(manifest.read_text())!=jobs:
            raise RuntimeError('Controlled ablation plan has changed')
    else:atomic_json(manifest,jobs)
    results=[]
    for i,job in enumerate(jobs,1):
        if time.perf_counter()-args.session_start>args.session_minutes*60:
            emit('[ABLATON SESSION BUDGET] Resume Colab to continue saved studies')
            break
        cfg=copy(args)
        cfg.output=output
        for key in ('seed','order','window_k','modes','depth','radius','basis_rank'):
            setattr(cfg,key,job[key])
        emit(f'[CLEAN ABLATION {i}/{len(jobs)}] {job["family"]}: '
             f'{job["method"]} order={job["order"]} K{job["window_k"]} '
             f'M{job["modes"]} D{job["depth"]} r{job["radius"]} '
             f'rank{job["basis_rank"]} seed{job["seed"]}')
        try:
            row=fit_one(U,cfg,job['ntrain'],job['method'],pm_width,pm_gap,device)
        except SessionBudgetExceeded:
            break
        results.append(dict(row,family=job['family'],order=job['order'],
                            modes=job['modes'],radius=job['radius'],
                            depth=job['depth'],basis_rank=job['basis_rank']))
    # Rebuild the cumulative table, not merely this session's completed runs.
    results=[]
    for job in jobs:
        cfg=copy(args)
        cfg.output=output
        for key in ('seed','order','window_k','modes','depth','radius','basis_rank'):
            setattr(cfg,key,job[key])
        m=make_model(job['method'],cfg,pm_width)
        config=record_config(cfg,m,job['ntrain'],job['method'],pm_width,pm_gap)
        file=model_folder(output,config)/'training_result.json'
        if file.exists():
            row=json.loads(file.read_text())
            results.append(dict(row,family=job['family'],order=job['order'],
                                modes=job['modes'],radius=job['radius'],
                                depth=job['depth'],basis_rank=job['basis_rank']))
    if results:
        import pandas as pd
        df=pd.DataFrame(results)
        df.to_csv(output/'clean_ablation_results.csv',index=False)
        df.to_json(output/'clean_ablation_results.json',orient='records',indent=2)
        # Join the reference AWFNO/TB-AWFNO N100 configurations for readable tables.
        reference_rows=[]
        for method in ('AWFNO','TB-AWFNO'):
            mm=make_model(method,args,pm_width)
            cc=record_config(args,mm,100,method,pm_width,pm_gap)
            fp=model_folder(args.output,cc)/'training_result.json'
            if fp.is_file():
                reference_rows.append(dict(json.loads(fp.read_text()),
                                           family='reference',order='GLL',
                                           modes=16,radius=2,depth=4,basis_rank=12))
        pd.DataFrame(reference_rows+results).to_csv(output/'clean_ablation_table_with_reference.csv',index=False)
        import matplotlib.pyplot as plt
        kinds=['ordering_and_components','temporal_window_K','Fourier_modes',
               'local_radius','network_depth','temporal_basis_rank','repeat_seed']
        baseline_rows={}
        for base_method in ('AWFNO','TB-AWFNO'):
            m=make_model(base_method,args,pm_width)
            c=record_config(args,m,100,base_method,pm_width,pm_gap)
            file=model_folder(args.output,c)/'training_result.json'
            if file.is_file():baseline_rows[base_method]=json.loads(file.read_text())
        fig,axs=plt.subplots(4,2,figsize=(15,16),constrained_layout=True)
        for ax,kind in zip(axs.flat,kinds):
            subset=[q for q in results if q['family']==kind]
            bmethod='TB-AWFNO' if kind=='temporal_basis_rank' else 'AWFNO'
            if bmethod in baseline_rows:
                reference=baseline_rows[bmethod]
                subset=[dict(reference,order='GLL',modes=16,radius=2,depth=4,
                             basis_rank=12)]+subset
            if kind=='repeat_seed':
                subset.extend(q for q in results
                              if q['family']=='ordering_and_components' and q['order']=='LGL')
            if kind=='ordering_and_components': labels=[q['order'] for q in subset]
            elif kind=='temporal_window_K':labels=[f"K{q['K']}" for q in subset]
            elif kind=='Fourier_modes':labels=[f"M{q['modes']}" for q in subset]
            elif kind=='local_radius':labels=[f"r{q['radius']}" for q in subset]
            elif kind=='network_depth':labels=[f"D{q['depth']}" for q in subset]
            elif kind=='temporal_basis_rank':labels=[f"rank{q['basis_rank']}" for q in subset]
            else: labels=[f"{q['order']} s{q['seed']}" for q in subset]
            ax.bar(labels,[q['best_validation_full_l2'] for q in subset])
            ax.set(title=kind,ylabel='Mean validation full physical L2')
            ax.tick_params(axis='x',rotation=25);ax.grid(axis='y',alpha=.2)
        axs.flat[-1].axis('off')
        fig.savefig(output/'clean_all_ablation_comparison.png',dpi=140)
        plt.close(fig)
    emit(f'[INDEPENDENT ABLATION STATUS] {len(results)}/{len(jobs)} completed; validation only.')
    return len(results)==len(jobs)


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--dataset',required=True,type=Path)
    p.add_argument('--output',required=True,type=Path)
    p.add_argument('--methods',nargs='+',default=list(MODELS),choices=MODELS)
    p.add_argument('--sizes',type=int,nargs='+',default=list(SIZES))
    p.add_argument('--seed',type=int,default=1234)
    p.add_argument('--width',type=int,default=32)
    p.add_argument('--modes',type=int,default=16)
    p.add_argument('--depth',type=int,default=4)
    p.add_argument('--radius',type=int,default=2)
    p.add_argument('--order',choices=('GLL','LGL','LLG'),default='GLL')
    p.add_argument('--window-k',type=int,default=20)
    p.add_argument('--basis-rank',type=int,default=12)
    p.add_argument('--window-stride',type=int,default=10)
    p.add_argument('--batch-size',type=int,default=8)
    p.add_argument('--eval-batch',type=int,default=3)
    p.add_argument('--max-epochs',type=int,default=45)
    p.add_argument('--min-epochs',type=int,default=12)
    p.add_argument('--patience',type=int,default=6)
    p.add_argument('--val-every',type=int,default=3)
    p.add_argument('--checkpoint-every',type=int,default=3)
    p.add_argument('--min-delta',type=float,default=1e-4)
    p.add_argument('--lr',type=float,default=0.0007)
    p.add_argument('--wd',type=float,default=1e-6)
    p.add_argument('--session-minutes',type=float,default=180)
    p.add_argument('--smoke-only',action='store_true')
    p.add_argument('--train-only',action='store_true')
    p.add_argument('--suite',choices=('full','comparison'),default='full')
    p.add_argument('--evaluate-test-only',action='store_true')
    p.add_argument('--verify-file-hash',action='store_true')
    p.add_argument('--force',action='store_true')
    p.add_argument('--force-test',action='store_true')
    args=p.parse_args()
    if args.train_only and args.evaluate_test_only:raise ValueError('Conflicting modes')
    if args.window_stride<1 or args.batch_size<1 or args.val_every<1:raise ValueError('Invalid batch/stride')
    if not (1<=args.basis_rank<=args.window_k<NT):raise ValueError('Invalid K/basis rank')
    if args.min_epochs>args.max_epochs:raise ValueError('Invalid min epochs')
    if not (set(args.methods)==set(MODELS) and set(args.sizes)==set(SIZES)) and not args.smoke_only:
        emit('[NOTE] Partial matrix requested: all requested runs must finish before requested test evaluation')
    args.output.mkdir(parents=True,exist_ok=True)
    args.session_start=time.perf_counter()
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    emit(f'[DEVICE] {device} | PyTorch {torch.__version__} | implementation {VERSION}')
    pm_width,pm_gap=smoke_test(args,device)
    if args.smoke_only:
        emit('[SMOKE ONLY] Did not open dataset, train or inspect test trajectories.')
        return
    U=load_source(args.dataset,verify_hash=args.verify_file_hash)
    emit(f'[DATASET] {tuple(U.shape)} | data SHA={EXPECTED_DATA_SHA} | '
         f'sealed 70 test trajectories')
    plan=expected_configs(args,pm_width,pm_gap)
    planfile=args.output/'predeclared_model_comparison.json'
    if planfile.exists():
        if json.loads(planfile.read_text())!=plan:
            raise RuntimeError('Existing plan differs. Use new output folder')
    else:atomic_json(planfile,plan)
    comparisons_complete=False
    ablations_complete=(args.suite=='comparison')
    if not args.evaluate_test_only:
        finished=[]
        for i,job in enumerate(plan,1):
            if time.perf_counter()-args.session_start>args.session_minutes*60:
                emit('[SESSION BUDGET] Rerun notebook to resume at next model')
                break
            emit(f'[EXPERIMENT {i}/{len(plan)}] {job["method"]} N{job["N_train"]}')
            try:
                row=fit_one(U,args,job['N_train'],job['method'],pm_width,pm_gap,device)
                finished.append(row)
            except SessionBudgetExceeded:
                break
            summarize_tables(args.output,finished,from_test=False)
        # Rebuild summary from all saved runs, including completed prior sessions.
        finished=[json.loads((Path(j['path'])/'training_result.json').read_text())
                  for j in plan if (Path(j['path'])/'training_result.json').exists()]
        summarize_tables(args.output,finished,from_test=False)
        comparisons_complete=(len(finished)==len(plan))
        emit(f'[TRAINING STATUS] {len(finished)}/{len(plan)} method comparisons completed; test not accessed')
        if comparisons_complete and args.suite=='full':
            ablations_complete=do_supplementary_ablations(U,args,pm_width,pm_gap,device)
        if args.train_only:return
    else:
        comparisons_complete=all((Path(j['path'])/'training_result.json').exists() for j in plan)
        if args.suite=='full':
            # Independent ablations are required for the predeclared full suite.
            folder=args.output/'independent_ablations'
            data=folder/'clean_ablation_results.json'
            ablations_complete=(data.is_file() and len(json.loads(data.read_text()))==20)
    if comparisons_complete and ablations_complete:
        emit('[TEST LOCK] All clean comparison and ablation configurations complete. Evaluating 70 sealed test trajectories.')
        evaluate_test(U,args,plan,device)
    else:
        emit('[TEST SEALED] Incomplete clean full suite. Rerun same notebook to resume.')


if __name__=='__main__':
    main()


## 3. Native additional-baseline models (byte-for-byte preserved)

The original 24-layer F-FNO, CNO, U-Net, and UNO-H10 architectures are all still implemented here, including their original normalization and checkpoint configuration. The **recovery launcher**, not this source file, excludes F-FNO from training.

In [ ]:
%%writefile /content/ns_new_baselines.py
#!/usr/bin/env python3
"""Add reference-oriented F-FNO, CNO, U-Net and UNO benchmarks to validated NS/Re500 setup.

Derived adaptions of the user's 1D Burgers and 2D Cahn-Hilliard reference-oriented
implementations (CNO simplified backbone, FourierFlow-style F-FNO, U-Net) and
UNO-H10 implementation, on a fully periodic 128^2 grid. Source architecture
attribution is in the accompanying notebook; these adaptations are NOT
claimed to be unmodified official original authors' training scripts.
"""
from __future__ import annotations
import argparse, copy, hashlib, json, math, os, random, time, sys, gc
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import weight_norm
from torch.utils.data import Dataset
import ns_base_original as base

NX=NY=base.NX
NT=base.NT
DT=base.DT
DX=base.DX
x_grid=np.arange(NX, dtype=np.float64)*DX
y_grid=np.arange(NY, dtype=np.float64)*DX
VERSION="ns-re500-reference-extra-baselines-h10-native-protocol-v3"
METHODS=("FFNO24-AR1", "CNO-DirectTime", "UNet-AR1", "UNO2D-H10")
SIZES=base.SIZES
CONFIG={
 "FFNO_WIDTH":64,"FFNO_LAYERS":24,"FFNO_MODES":16,"FFNO_FACTOR":4,
 "FFNO_FF_LAYERS":2,"FFNO_SHARE_FOURIER_WEIGHT":True,"FFNO_WEIGHT_NORM":True,
 "FFNO_FOURIER_INIT_GAIN":0.1,"FFNO_OUT_STEPS":1,
 "CNO_LAYERS":3,"CNO_CHANNEL_MULTIPLIER":32,"CNO_RES_BLOCKS":3,
 "CNO_NECK_BLOCKS":3,"CNO_USE_BATCH_NORM":True,
 "UNET_BASE_CHANNELS":32,"UNET_LEVELS":4,"UNET_NORM":"group",
 "UNET_PADDING_MODE":"circular",
 "WIDTH":32,"FACTOR":0.75,"HISTORY_LENGTH":10,
 "MODES":[11,7,3,3,3,7,11],"NORMALIZE_BLOCKS":False,
 "UNO_TRAIN_ROLLOUT_STEPS":40,"UNO_TRAIN_WINDOW_STRIDE":40,
}
# Use four sine/cosine coordinates (periodic on [0,2*pi)^2).
def normalized_grid_channels(x_values,y_values):
    return base.periodic_features('cpu')[0, :4]



class WNLinear(nn.Linear):
    def __init__(self, in_features, out_features, use_weight_norm):
        super().__init__(in_features, out_features, bias=True)
        if use_weight_norm:
            weight_norm(self)


class FeedForward(nn.Module):
    def __init__(
        self,
        dim,
        factor,
        n_layers,
        use_weight_norm,
        dropout=0.0,
    ):
        super().__init__()
        if n_layers < 1:
            raise ValueError("n_layers must be positive.")
        layers = []
        for index in range(n_layers):
            in_dim = dim if index == 0 else dim * factor
            out_dim = dim if index == n_layers - 1 else dim * factor
            layers.append(WNLinear(in_dim, out_dim, use_weight_norm))
            layers.append(nn.Dropout(dropout))
            layers.append(
                nn.ReLU(inplace=True)
                if index < n_layers - 1
                else nn.Identity()
            )
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)


class FactorizedSpectralLayer2D(nn.Module):
    def __init__(
        self,
        width,
        modes,
        factor,
        n_ff_layers,
        use_weight_norm,
    ):
        super().__init__()
        self.width = int(width)
        self.modes = int(modes)
        self.backcast_ff = FeedForward(
            dim=width,
            factor=factor,
            n_layers=n_ff_layers,
            use_weight_norm=use_weight_norm,
            dropout=0.0,
        )

    def forward_fourier(self, x, fourier_weights):
        # x: (B, Nx, Ny, C), matching FourierFlow's grid_2d.py.
        x_channels = x.permute(0, 3, 1, 2).contiguous()
        batch, _, nx, ny = x_channels.shape

        # Transform along y.
        x_ft_y = torch.fft.rfft(x_channels, dim=-1, norm="ortho")
        out_ft_y = torch.zeros(
            batch,
            self.width,
            nx,
            ny // 2 + 1,
            dtype=x_ft_y.dtype,
            device=x.device,
        )
        retained_y = min(self.modes, ny // 2 + 1)
        weight_y = torch.view_as_complex(
            fourier_weights[0][:, :, :retained_y, :].contiguous()
        )
        out_ft_y[:, :, :, :retained_y] = torch.einsum(
            "bixy,ioy->boxy",
            x_ft_y[:, :, :, :retained_y],
            weight_y,
        )
        transformed_y = torch.fft.irfft(
            out_ft_y, n=ny, dim=-1, norm="ortho"
        )

        # Transform along x.
        x_ft_x = torch.fft.rfft(x_channels, dim=-2, norm="ortho")
        out_ft_x = torch.zeros(
            batch,
            self.width,
            nx // 2 + 1,
            ny,
            dtype=x_ft_x.dtype,
            device=x.device,
        )
        retained_x = min(self.modes, nx // 2 + 1)
        weight_x = torch.view_as_complex(
            fourier_weights[1][:, :, :retained_x, :].contiguous()
        )
        out_ft_x[:, :, :retained_x, :] = torch.einsum(
            "bixy,iox->boxy",
            x_ft_x[:, :, :retained_x, :],
            weight_x,
        )
        transformed_x = torch.fft.irfft(
            out_ft_x, n=nx, dim=-2, norm="ortho"
        )

        transformed = transformed_x + transformed_y
        return transformed.permute(0, 2, 3, 1).contiguous()

    def forward(self, x, fourier_weights):
        return self.backcast_ff(self.forward_fourier(x, fourier_weights))


class FFNO2D_AR(nn.Module):
    def __init__(self, config):
        super().__init__()
        width = int(config["FFNO_WIDTH"])
        modes = int(config["FFNO_MODES"])
        self.n_layers = int(config["FFNO_LAYERS"])
        use_weight_norm = bool(config["FFNO_WEIGHT_NORM"])

        coordinates = normalized_grid_channels(x_grid, y_grid)
        self.register_buffer(
            "coordinates",
            coordinates.unsqueeze(0),
            persistent=False,
        )

        # State plus x and y coordinate channels.
        self.in_proj = WNLinear(5, width, use_weight_norm)

        self.fourier_weights = nn.ParameterList()
        for _ in range(2):
            parameter = nn.Parameter(
                torch.empty(width, width, modes, 2)
            )
            nn.init.xavier_normal_(
                parameter,
                gain=float(config["FFNO_FOURIER_INIT_GAIN"]),
            )
            self.fourier_weights.append(parameter)

        self.layers = nn.ModuleList(
            [
                FactorizedSpectralLayer2D(
                    width=width,
                    modes=modes,
                    factor=int(config["FFNO_FACTOR"]),
                    n_ff_layers=int(config["FFNO_FF_LAYERS"]),
                    use_weight_norm=use_weight_norm,
                )
                for _ in range(self.n_layers)
            ]
        )
        self.out = nn.Sequential(
            WNLinear(width, 128, use_weight_norm),
            WNLinear(128, int(config["FFNO_OUT_STEPS"]), use_weight_norm),
        )

    def forward(self, normalized_state):
        if normalized_state.ndim != 3:
            raise ValueError(
                f"Expected (B,Nx,Ny), received {normalized_state.shape}."
            )
        # This second guard is deliberate. F-FNO is already excluded by
        # amp_enabled_for(), but forcing the complete F-FNO forward pass to
        # float32 also protects against a stale or enclosing autocast context.
        # CUDA does not implement the ComplexHalf einsum used below.
        with torch.autocast(
            device_type=normalized_state.device.type,
            enabled=False,
        ):
            return self._forward_float32(normalized_state.float())

    def _forward_float32(self, normalized_state):
        batch = normalized_state.shape[0]
        coordinates = (
            self.coordinates.expand(batch, -1, -1, -1)
            .permute(0, 2, 3, 1)
        )
        x = torch.cat(
            [normalized_state.unsqueeze(-1), coordinates], dim=-1
        )
        x = self.in_proj(x)

        backcast = None
        for layer in self.layers:
            backcast = layer(x, self.fourier_weights)
            x = x + backcast
        return self.out(backcast).permute(0, 3, 1, 2).contiguous()


class CNO_LReLU(nn.Module):
    def __init__(self, in_size, out_size):
        super().__init__()
        self.in_size = int(in_size)
        self.out_size = int(out_size)
        self.activation = nn.LeakyReLU()

    def forward(self, x):
        x = F.interpolate(
            x,
            size=(2 * self.in_size, 2 * self.in_size),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )
        x = self.activation(x)
        return F.interpolate(
            x,
            size=(self.out_size, self.out_size),
            mode="bicubic",
            align_corners=False,
            antialias=True,
        )


class CNOBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        in_size,
        out_size,
        use_bn=True,
    ):
        super().__init__()
        self.convolution = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3, padding=1, padding_mode="circular",
        )
        self.batch_norm = (
            nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()
        )
        self.activation = CNO_LReLU(in_size, out_size)

    def forward(self, x):
        return self.activation(self.batch_norm(self.convolution(x)))


class CNOLiftProjectBlock(nn.Module):
    def __init__(self, in_channels, out_channels, size, latent_dim=64):
        super().__init__()
        self.intermediate = CNOBlock(
            in_channels,
            latent_dim,
            in_size=size,
            out_size=size,
            use_bn=False,
        )
        self.convolution = nn.Conv2d(
            latent_dim,
            out_channels,
            kernel_size=3, padding=1, padding_mode="circular",
        )

    def forward(self, x):
        return self.convolution(self.intermediate(x))


class CNOResidualBlock(nn.Module):
    def __init__(self, channels, size, use_bn=True):
        super().__init__()
        self.convolution1 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.convolution2 = nn.Conv2d(
            channels, channels, kernel_size=3, padding=1, padding_mode="circular"
        )
        self.batch_norm1 = (
            nn.BatchNorm2d(channels) if use_bn else nn.Identity()
        )
        self.batch_norm2 = (
            nn.BatchNorm2d(channels) if use_bn else nn.Identity()
        )
        self.activation = CNO_LReLU(size, size)

    def forward(self, x):
        residual = self.convolution1(x)
        residual = self.batch_norm1(residual)
        residual = self.activation(residual)
        residual = self.convolution2(residual)
        residual = self.batch_norm2(residual)
        return x + residual


class CNOResNet(nn.Module):
    def __init__(self, channels, size, num_blocks, use_bn=True):
        super().__init__()
        self.blocks = nn.Sequential(
            *[
                CNOResidualBlock(channels, size, use_bn)
                for _ in range(int(num_blocks))
            ]
        )

    def forward(self, x):
        return self.blocks(x)


class CNO2D_DirectTime(nn.Module):
    def __init__(self, config):
        super().__init__()
        n_layers = int(config["CNO_LAYERS"])
        channel_multiplier = int(config["CNO_CHANNEL_MULTIPLIER"])
        n_res = int(config["CNO_RES_BLOCKS"])
        n_res_neck = int(config["CNO_NECK_BLOCKS"])
        use_bn = bool(config["CNO_USE_BATCH_NORM"])
        size = NX
        if NX != NY or size % (2**n_layers) != 0:
            raise ValueError("CNO expects a divisible square grid.")

        self.n_layers = n_layers
        self.register_buffer(
            "coordinates",
            normalized_grid_channels(x_grid, y_grid).unsqueeze(0),
            persistent=False,
        )

        lift_dim = channel_multiplier // 2
        encoder_features = [lift_dim]
        for index in range(n_layers):
            encoder_features.append(
                (2**index) * channel_multiplier
            )
        decoder_features_in = list(reversed(encoder_features[1:]))
        decoder_features_out = list(reversed(encoder_features[:-1]))
        for index in range(1, n_layers):
            decoder_features_in[index] *= 2

        encoder_sizes = [size // (2**i) for i in range(n_layers + 1)]
        decoder_sizes = [
            size // (2 ** (n_layers - i))
            for i in range(n_layers + 1)
        ]

        # u0, x, y, query time.
        self.lift = CNOLiftProjectBlock(6, lift_dim, size=size)
        self.project = CNOLiftProjectBlock(
            lift_dim + decoder_features_out[-1],
            1,
            size=size,
        )

        self.encoder = nn.ModuleList(
            [
                CNOBlock(
                    encoder_features[i],
                    encoder_features[i + 1],
                    encoder_sizes[i],
                    encoder_sizes[i + 1],
                    use_bn,
                )
                for i in range(n_layers)
            ]
        )
        self.skip_expansion = nn.ModuleList(
            [
                CNOBlock(
                    encoder_features[i],
                    encoder_features[i],
                    encoder_sizes[i],
                    decoder_sizes[n_layers - i],
                    use_bn,
                )
                for i in range(n_layers + 1)
            ]
        )
        self.decoder = nn.ModuleList(
            [
                CNOBlock(
                    decoder_features_in[i],
                    decoder_features_out[i],
                    decoder_sizes[i],
                    decoder_sizes[i + 1],
                    use_bn,
                )
                for i in range(n_layers)
            ]
        )
        self.res_nets = nn.ModuleList(
            [
                CNOResNet(
                    encoder_features[level],
                    encoder_sizes[level],
                    n_res,
                    use_bn,
                )
                for level in range(n_layers)
            ]
        )
        self.neck = CNOResNet(
            encoder_features[n_layers],
            encoder_sizes[n_layers],
            n_res_neck,
            use_bn,
        )

    def forward(self, normalized_u0, query_time):
        if normalized_u0.ndim != 3:
            raise ValueError(normalized_u0.shape)
        batch = normalized_u0.shape[0]
        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        time_channel = query_time.reshape(batch, 1, 1, 1).expand(
            -1, 1, NX, NY
        )
        x = torch.cat(
            [normalized_u0.unsqueeze(1), coordinates, time_channel], dim=1
        )
        x = self.lift(x)
        skip = []
        for index in range(self.n_layers):
            skip_value = self.res_nets[index](x)
            skip.append(skip_value)
            # This follows the official simplified implementation exactly:
            # the down block acts on x; the ResNet value forms the skip.
            x = self.encoder[index](x)
        x = self.neck(x)

        for index in range(self.n_layers):
            if index == 0:
                x = self.skip_expansion[self.n_layers](x)
            else:
                expanded_skip = self.skip_expansion[
                    self.n_layers - index
                ](skip[-index])
                x = torch.cat([x, expanded_skip], dim=1)
            x = self.decoder[index](x)

        x = torch.cat([x, self.skip_expansion[0](skip[0])], dim=1)
        return self.project(x).squeeze(1)


def group_count(channels):
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class UNetConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, config):
        super().__init__()
        padding_mode = config["UNET_PADDING_MODE"]
        norm_kind = config["UNET_NORM"]

        def normalization(channels):
            if norm_kind == "group":
                return nn.GroupNorm(group_count(channels), channels)
            if norm_kind == "batch":
                return nn.BatchNorm2d(channels)
            if norm_kind == "none":
                return nn.Identity()
            raise ValueError(norm_kind)

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=padding_mode,
            ),
            normalization(out_channels),
            nn.GELU(),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                padding_mode=padding_mode,
            ),
            normalization(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class UNet2D_AR(nn.Module):
    def __init__(self, config):
        super().__init__()
        base = int(config["UNET_BASE_CHANNELS"])
        levels = int(config["UNET_LEVELS"])
        if levels != 4:
            raise ValueError("This audited implementation uses four levels.")
        channels = [base, 2 * base, 4 * base, 8 * base, 16 * base]
        self.register_buffer(
            "coordinates",
            normalized_grid_channels(x_grid, y_grid).unsqueeze(0),
            persistent=False,
        )

        self.encoders = nn.ModuleList(
            [
                UNetConvBlock(5, channels[0], config),
                UNetConvBlock(channels[0], channels[1], config),
                UNetConvBlock(channels[1], channels[2], config),
                UNetConvBlock(channels[2], channels[3], config),
            ]
        )
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = UNetConvBlock(
            channels[3], channels[4], config
        )
        self.upconvs = nn.ModuleList(
            [
                nn.ConvTranspose2d(
                    channels[4], channels[3], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[3], channels[2], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[2], channels[1], kernel_size=2, stride=2
                ),
                nn.ConvTranspose2d(
                    channels[1], channels[0], kernel_size=2, stride=2
                ),
            ]
        )
        self.decoders = nn.ModuleList(
            [
                UNetConvBlock(2 * channels[3], channels[3], config),
                UNetConvBlock(2 * channels[2], channels[2], config),
                UNetConvBlock(2 * channels[1], channels[1], config),
                UNetConvBlock(2 * channels[0], channels[0], config),
            ]
        )
        self.output = nn.Conv2d(channels[0], 1, kernel_size=1)

    def forward(self, normalized_state):
        batch = normalized_state.shape[0]
        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        x = torch.cat([normalized_state.unsqueeze(1), coordinates], dim=1)
        skip = []
        for encoder in self.encoders:
            x = encoder(x)
            skip.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for index, (upconv, decoder) in enumerate(
            zip(self.upconvs, self.decoders)
        ):
            x = upconv(x)
            x = torch.cat([x, skip[-(index + 1)]], dim=1)
            x = decoder(x)
        return self.output(x).squeeze(1)




class SpectralConv2dUNO(nn.Module):
    def __init__(
        self, in_channels, out_channels, dim1, dim2, modes1, modes2
    ):
        super().__init__()
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)
        self.modes1 = int(modes1)
        self.modes2 = int(modes2)
        scale = (1.0 / (2.0 * self.in_channels)) ** 0.5
        shape = (
            self.in_channels,
            self.out_channels,
            self.modes1,
            self.modes2,
        )
        self.weights1 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )
        self.weights2 = nn.Parameter(
            scale * torch.randn(*shape, dtype=torch.cfloat)
        )

    @staticmethod
    def complex_multiply(x, weights):
        return torch.einsum("bixy,ioxy->boxy", x, weights)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        max_modes1 = min(x.shape[-2] // 2, output_dim1 // 2)
        max_modes2 = min(x.shape[-1] // 2 + 1, output_dim2 // 2 + 1)
        if self.modes1 > max_modes1 or self.modes2 > max_modes2:
            raise ValueError(
                f"Modes {(self.modes1, self.modes2)} exceed the admissible "
                f"limit {(max_modes1, max_modes2)} for "
                f"{tuple(x.shape[-2:])} -> {(output_dim1, output_dim2)}."
            )

        # FFT operations remain in float32/complex64; AMP is disabled.
        x_ft = torch.fft.rfft2(x.float(), norm="forward")
        out_ft = torch.zeros(
            x.shape[0],
            self.out_channels,
            output_dim1,
            output_dim2 // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )
        out_ft[:, :, : self.modes1, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, : self.modes1, : self.modes2],
                self.weights1,
            )
        )
        out_ft[:, :, -self.modes1 :, : self.modes2] = (
            self.complex_multiply(
                x_ft[:, :, -self.modes1 :, : self.modes2],
                self.weights2,
            )
        )
        return torch.fft.irfft2(
            out_ft,
            s=(output_dim1, output_dim2),
            norm="forward",
        )


class PointwiseOp2D(nn.Module):
    def __init__(self, in_channels, out_channels, dim1, dim2):
        super().__init__()
        self.conv = nn.Conv2d(int(in_channels), int(out_channels), 1)
        self.dim1 = int(dim1)
        self.dim2 = int(dim2)

    def forward(self, x, dim1=None, dim2=None):
        output_dim1 = self.dim1 if dim1 is None else int(dim1)
        output_dim2 = self.dim2 if dim2 is None else int(dim2)
        x = self.conv(x)
        return F.interpolate(
            x,
            size=(output_dim1, output_dim2),
            mode="bicubic",
            align_corners=True,
            antialias=True,
        )


class OperatorBlock2D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        dim1,
        dim2,
        modes1,
        modes2,
        normalize=False,
        nonlinearity=True,
    ):
        super().__init__()
        self.conv = SpectralConv2dUNO(
            in_channels, out_channels, dim1, dim2, modes1, modes2
        )
        self.w = PointwiseOp2D(
            in_channels, out_channels, dim1, dim2
        )
        self.normalize = bool(normalize)
        self.nonlinearity = bool(nonlinearity)
        if self.normalize:
            self.normalizer = nn.InstanceNorm2d(
                int(out_channels), affine=True
            )

    def forward(self, x, dim1=None, dim2=None):
        x = self.conv(x, dim1, dim2) + self.w(x, dim1, dim2)
        if self.normalize:
            x = self.normalizer(x)
        if self.nonlinearity:
            x = F.gelu(x)
        return x


class UNO2DHistory(nn.Module):
    """History-to-one-state UNO2D used autoregressively."""

    def __init__(self, x_coordinates, y_coordinates, config):
        super().__init__()
        width = int(config["WIDTH"])
        factor = float(config["FACTOR"])
        history = int(config["HISTORY_LENGTH"])
        modes = list(config["MODES"])
        normalize = bool(config["NORMALIZE_BLOCKS"])
        if len(modes) != 7:
            raise ValueError(modes)

        # Official periodic coordinate embedding: sin/cos in x and y.
        x_phase = 2.0 * np.pi * (
            (x_coordinates - x_coordinates[0])
            / (NX * DX)
        )
        y_phase = 2.0 * np.pi * (
            (y_coordinates - y_coordinates[0])
            / (NY * DX)
        )
        x_phase, y_phase = np.meshgrid(
            x_phase, y_phase, indexing="ij"
        )
        coordinates = np.stack(
            [
                np.sin(x_phase),
                np.sin(y_phase),
                np.cos(x_phase),
                np.cos(y_phase),
            ],
            axis=-1,
        ).astype(np.float32)
        self.register_buffer(
            "coordinates", torch.from_numpy(coordinates).unsqueeze(0)
        )

        self.history_length = history
        self.fc = nn.Linear(history + 4, width // 2)
        self.fc0 = nn.Linear(width // 2, width)

        c0 = width
        c1 = int(2 * factor * width)
        c2 = int(4 * factor * width)
        c3 = int(8 * factor * width)
        self.L0 = OperatorBlock2D(
            c0, c1, 96, 96, modes[0], modes[0], normalize
        )
        self.L1 = OperatorBlock2D(
            c1, c2, 64, 64, modes[1], modes[1], normalize
        )
        self.L2 = OperatorBlock2D(
            c2, c3, 32, 32, modes[2], modes[2], normalize
        )
        self.L3 = OperatorBlock2D(
            c3, c3, 32, 32, modes[3], modes[3], normalize
        )
        self.L4 = OperatorBlock2D(
            c3, c2, 64, 64, modes[4], modes[4], normalize
        )
        self.L5 = OperatorBlock2D(
            2 * c2, c1, 96, 96, modes[5], modes[5], normalize
        )
        self.L6 = OperatorBlock2D(
            2 * c1, width, 128, 128, modes[6], modes[6], normalize
        )
        self.fc1 = nn.Linear(2 * width, 4 * width)
        self.fc2 = nn.Linear(4 * width, 1)

    def forward(self, history):
        if history.ndim != 4:
            raise ValueError(
                f"Expected (batch,x,y,history), received {history.shape}."
            )
        batch, nx, ny, history_length = history.shape
        if (nx, ny) != (NX, NY):
            raise ValueError(f"Expected {NX}x{NY}, received {(nx, ny)}.")
        if history_length != self.history_length:
            raise ValueError(
                f"Expected history {self.history_length}, "
                f"received {history_length}."
            )

        coordinates = self.coordinates.expand(batch, -1, -1, -1)
        x = torch.cat([history, coordinates], dim=-1)
        x_fc = F.gelu(self.fc(x))
        x_fc0 = F.gelu(self.fc0(x_fc)).permute(0, 3, 1, 2)

        x_c0 = self.L0(x_fc0, 96, 96)
        x_c1 = self.L1(x_c0, 64, 64)
        x_c2 = self.L2(x_c1, 32, 32)
        x_c3 = self.L3(x_c2, 32, 32)
        x_c4 = self.L4(x_c3, 64, 64)
        x_c4 = torch.cat([x_c4, x_c1], dim=1)
        x_c5 = self.L5(x_c4, 96, 96)
        x_c5 = torch.cat([x_c5, x_c0], dim=1)
        x_c6 = self.L6(x_c5, 128, 128)
        x_c6 = torch.cat([x_c6, x_fc0], dim=1)
        x_c6 = x_c6.permute(0, 2, 3, 1).contiguous()
        return self.fc2(F.gelu(self.fc1(x_c6))).squeeze(-1)






# --- Original-method benchmark-compatible training, validation and test protocol ---
class AdjacentPairs(Dataset):
    def __init__(self,U,ids):
        self.U=U;self.ids=np.asarray(ids,dtype=int)
    def __len__(self):return len(self.ids)*(NT-1)
    def __getitem__(self,j):
        i=self.ids[j//(NT-1)];t=j%(NT-1)
        u=self.U[int(i)]
        return u[...,t].unsqueeze(0),u[...,t+1].unsqueeze(0)

class UNO40StepRolloutWindows(Dataset):
    """Native UNO-H10 protocol adapted from the previously verified Cahn-Hilliard UNO notebook.

    Each sample supplies 10 exact training history fields followed by 40 targets.
    The model runs all 40 targets autoregressively with gradients through the rollout.
    At test time it only observes the first 10 fields, then free-runs to t100.
    """
    def __init__(self,U,ids,history=10,horizon=40,stride=40):
        self.U=U;self.ids=np.asarray(ids,dtype=int)
        self.history=int(history);self.horizon=int(horizon)
        last=NT-self.history-self.horizon
        if last<0:raise ValueError('Too few frames for native UNO-H10 training')
        self.starts=list(range(0,last+1,int(stride)))
        if self.starts[-1]!=last:self.starts.append(last)
    def __len__(self):return len(self.ids)*len(self.starts)
    def __getitem__(self,j):
        i=int(self.ids[j//len(self.starts)])
        t=self.starts[j%len(self.starts)]
        u=self.U[i]
        return (u[...,t:t+self.history].permute(2,0,1).contiguous(),
                u[...,t+self.history:t+self.history+self.horizon].permute(2,0,1).contiguous())


class DirectTimeSamples(Dataset):
    def __init__(self,U,ids):
        self.U=U;self.ids=np.asarray(ids,dtype=int)
    def __len__(self):return len(self.ids)*(NT-1)
    def __getitem__(self,j):
        i=self.ids[j//(NT-1)];t=j%(NT-1)+1
        u=self.U[int(i)]
        xt=torch.stack((u[...,0],torch.full((NX,NY),float(t)/(NT-1))))
        return xt,u[...,t].unsqueeze(0)


def make_method(method):
    if method=='FFNO24-AR1':return FFNO2D_AR(CONFIG)
    if method=='CNO-DirectTime':return CNO2D_DirectTime(CONFIG)
    if method=='UNet-AR1':return UNet2D_AR(CONFIG)
    if method=='UNO2D-H10':return UNO2DHistory(x_grid,y_grid,CONFIG)
    raise ValueError(method)


def call_method(model,method,x):
    # x is normalized state B,1,NX,NX (except CNO B,2,NX,NX, time channel left unnormalized).
    if method=='FFNO24-AR1':return model(x[:,0])
    if method=='UNet-AR1':return model(x[:,0]).unsqueeze(1)
    if method=='UNO2D-H10':return model(x.permute(0,2,3,1)).unsqueeze(1)
    if method=='CNO-DirectTime':return model(x[:,0],x[:,1,0,0]).unsqueeze(1)
    raise ValueError(method)


def encode_batch(batch,method,device,mean,std):
    x,y=batch
    x=x.to(device,non_blocking=True).float()
    y=(y.to(device,non_blocking=True).float()-mean)/std
    if method=='CNO-DirectTime':
        x=torch.cat(((x[:,:1]-mean)/std,x[:,1:2]),dim=1)
    else:x=(x-mean)/std
    return x,y


def predict_complete(model,method,U,ids,device,mean,std,batch_size=2):
    model.eval();out=[];true=[]
    with torch.inference_mode():
        for p in range(0,len(ids),batch_size):
            ids_p=ids[p:p+batch_size]
            chunk=torch.stack([U[int(i)] for i in ids_p],dim=0)
            initial=(chunk[...,0].unsqueeze(1).to(device)-mean)/std
            parts=[initial]
            if method in ('FFNO24-AR1', 'UNet-AR1'):
                # Previously established native autoregressive one-step baselines.
                state=initial
                for _ in range(NT-1):
                    state=call_method(model,method,state)
                    parts.append(state)
            elif method=='UNO2D-H10':
                history=(chunk[...,:10].permute(0,3,1,2).to(device)-mean)/std
                parts=[history[:,k:k+1] for k in range(10)]
                for _ in range(10,NT):
                    z=call_method(model,method,history)
                    parts.append(z)
                    history=torch.cat((history[:,1:],z),dim=1)
            elif method=='CNO-DirectTime':
                # Genuine direct-time initial-condition queries: no preceding predictions.
                for k in range(1,NT):
                    time_feature=torch.full_like(initial,float(k)/(NT-1))
                    z=call_method(model,method,torch.cat((initial,time_feature),dim=1))
                    parts.append(z)
            y=torch.cat(parts,dim=1)
            assert y.shape==(len(ids_p),NT,NX,NY),(method,y.shape)
            out.append((y*std+mean).float().cpu().numpy())
            true.append(chunk.permute(0,3,1,2).numpy())
    return np.concatenate(out),np.concatenate(true)


def forecast_from_t10_l2(pred,true):
    """Absolute physical space-time L2 on common, fully forecast interval [t10,t100]."""
    w=np.ones(NT-10,dtype=np.float64);w[0]=w[-1]=0.5
    err=(pred[:,10:].astype(np.float64)-true[:,10:].astype(np.float64))
    sq=np.sum(err*err,axis=(-1,-2))*DX*DX
    return np.sqrt(DT*np.sum(sq*w[None,:],axis=-1))


def physical_relative_h1(pred,true):
    # periodic torus: centered physical derivatives; gradients only, no amplitude mix.
    p=pred.astype(np.float64);t=true.astype(np.float64)
    def grad(v):
        return (np.roll(v,-1,axis=-2)-np.roll(v,1,axis=-2))/(2*DX),\
               (np.roll(v,-1,axis=-1)-np.roll(v,1,axis=-1))/(2*DX)
    gx,gy=grad(p-t);ux,uy=grad(t)
    en=np.sum(gx*gx+gy*gy,axis=(-2,-1))*DX*DX
    dn=np.sum(ux*ux+uy*uy,axis=(-2,-1))*DX*DX
    tw=np.ones(NT);tw[0]=tw[-1]=.5
    return {
      'full_relative_h1_semi':np.sqrt((en*tw[None]).sum(1)/np.maximum((dn*tw[None]).sum(1),1e-30)),
      'final_relative_h1_semi':np.sqrt(en[:,-1]/np.maximum(dn[:,-1],1e-30)),
    }


def cfg_for(args,method,n):
    return dict(version=VERSION,dataset_sha256=base.EXPECTED_DATA_SHA,
      method=method,N_train=n,seed=args.seed,model_config=CONFIG,
      epochs=args.max_epochs,patience=args.patience,min_epochs=args.min_epochs,
      val_every=args.val_every,batch_size=args.batch_size,
      micro_batch_cno=args.micro_batch_cno,
      lr=args.lr,wd=args.wd,uno_train_steps=40,uno_train_stride=40,
      ffno_steps=1,
      selection='validation_mean_absolute_physical_L2_Omega_times_T',
      zero_test_use_in_selection=True)


def run_epoch(model,method,dataset,opt,device,mean,std,epoch,seed,effective_batch,micro_batch):
    model.train()
    indices=torch.randperm(len(dataset),generator=torch.Generator().manual_seed(seed+1000003*epoch)).tolist()
    steps=0;total=0.;acc_samples=0
    opt.zero_grad(set_to_none=True)
    for start in range(0,len(indices),micro_batch):
        selected=indices[start:start+micro_batch]
        pairs=[dataset[i] for i in selected]
        batch=torch.stack([q[0] for q in pairs]),torch.stack([q[1] for q in pairs])
        x,y=encode_batch(batch,method,device,mean,std)
        if method=='UNO2D-H10':
            # The prior UNO-H10 benchmark uses a 40-step differentiable rollout.
            # Checkpoint activations at each step to control 128^2 GPU memory.
            from torch.utils.checkpoint import checkpoint
            history=x
            horizon=y.shape[1]
            assert horizon==CONFIG['UNO_TRAIN_ROLLOUT_STEPS']
            loss=torch.zeros((),device=device,dtype=torch.float32)
            for k in range(horizon):
                prediction=checkpoint(lambda state:call_method(model,method,state),
                                      history,use_reentrant=False)
                target=y[:,k:k+1]
                # Normalized teacher targets are used ONLY in the loss.
                # Feedback always uses model output; no teacher forcing within unroll.
                loss=loss+F.mse_loss(prediction,target)/horizon
                history=torch.cat((history[:,1:],prediction),dim=1)
        else:
            pred=call_method(model,method,x)
            assert pred.shape==y.shape,(method,pred.shape,y.shape)
            loss=F.mse_loss(pred,y)
        if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite train loss {method}')
        (loss*(len(selected)/effective_batch)).backward()
        acc_samples+=len(selected)
        total+=float(loss.detach())*len(selected)
        if acc_samples>=effective_batch or start+micro_batch>=len(indices):
            if acc_samples!=effective_batch:
                for p in model.parameters():
                    if p.grad is not None:p.grad.mul_(effective_batch/acc_samples)
            grad=torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            if not torch.isfinite(grad):raise FloatingPointError(f'Nonfinite gradients in {method}')
            opt.step();opt.zero_grad(set_to_none=True)
            steps+=1;acc_samples=0
    return total/len(dataset),steps


def folder_for(args,method,n):
    cfg=cfg_for(args,method,n)
    return args.output/'runs'/method/f'N{n}_seed{args.seed}_{base.hash_config(cfg)}'


def save_train_checkpoint(folder,model,opt,epoch,best,best_epoch,best_state,history,elapsed,steps,cfg):
    tmp=folder/'checkpoint.tmp'
    torch.save(dict(model=model.state_dict(),optimizer=opt.state_dict(),epoch=epoch,
                    best=best,best_epoch=best_epoch,best_state=best_state,
                    history=history,elapsed=elapsed,steps=steps,config=cfg,
                    torch_rng=torch.get_rng_state(),numpy_rng=np.random.get_state(),
                    python_rng=random.getstate(),
                    cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None),tmp)
    tmp.replace(folder/'checkpoint.pt')


def fit_model(U,args,n,method,device):
    ids=np.arange(n);valids=np.arange(200,230)
    folder=folder_for(args,method,n);folder.mkdir(parents=True,exist_ok=True)
    cfg=cfg_for(args,method,n)
    done=folder/'training_result.json'
    if done.exists():return json.loads(done.read_text())
    mean,std=base.load_normalized_stats(U,ids,args.original_root)
    base.seed_all(args.seed)
    model=make_method(method).to(device)
    opt=torch.optim.AdamW(model.parameters(),lr=args.lr,weight_decay=args.wd)
    if method=='UNO2D-H10':
        dataset=UNO40StepRolloutWindows(U,ids,history=10,
                    horizon=CONFIG['UNO_TRAIN_ROLLOUT_STEPS'],
                    stride=CONFIG['UNO_TRAIN_WINDOW_STRIDE'])
    elif method=='CNO-DirectTime':dataset=DirectTimeSamples(U,ids)
    else:dataset=AdjacentPairs(U,ids)
    # Differentiable UNO unrolling is much more memory-intensive than one-step training.
    # Avoid conflating effective batch (8) with physical GPU microbatch (1).
    micro=min(args.batch_size,1 if method in ('UNO2D-H10','FFNO24-AR1','CNO-DirectTime') else args.micro_batch_other)

    epoch_start=0;best=float('inf');best_epoch=0;best_state=None;history=[];steps=0;prev=0;bad=0
    saved=folder/'checkpoint.pt'
    if saved.exists():
        st=torch.load(saved,map_location='cpu',weights_only=False)
        if st['config']!=cfg:raise ValueError(f'Mismatched checkpoint {saved}')
        model.load_state_dict(st['model']);opt.load_state_dict(st['optimizer'])
        epoch_start=st['epoch'];best=st['best'];best_epoch=st['best_epoch']
        best_state=st['best_state'];history=st['history'];steps=st['steps'];prev=st['elapsed']
        torch.set_rng_state(st['torch_rng']);np.random.set_state(st['numpy_rng']);random.setstate(st['python_rng'])
        if st['cuda_rng'] is not None and device.type=='cuda':torch.cuda.set_rng_state_all(st['cuda_rng'])
        for entry in reversed(history):
            if np.isfinite(entry['val_full_l2']):
                if entry['epoch']==best_epoch:break
                bad+=1
        base.emit(f'[RESUME] {method} N={n} epoch {epoch_start}')
    base.emit(f'[FIT] {method} N{n}: params={base.param_count(model):,}, real={base.real_dof(model):,}, '
              f'train samples/epoch={len(dataset)}, microbatch={micro}, effective batch={args.batch_size}')
    start=time.perf_counter()
    for epoch in range(epoch_start+1,args.max_epochs+1):
        train_mse,ns=run_epoch(model,method,dataset,opt,device,mean,std,epoch,args.seed,args.batch_size,micro)
        steps+=ns
        val_full=val_final=float('nan')
        if epoch==1 or epoch%args.val_every==0 or epoch==args.max_epochs:
            pred,true=predict_complete(model,method,U,valids,device,mean,std,
                                       1 if method=='CNO-DirectTime' else args.eval_batch)
            m=base.l2_metrics(pred,true)
            val_full=float(np.mean(m['full']));val_final=float(np.mean(m['final']))
            if val_full<best-args.min_delta:
                best=val_full;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                bad=0
            else:bad+=1
            del pred,true
        history.append(dict(epoch=epoch,train_mse=train_mse,val_full_l2=val_full,
                            val_final_l2=val_final,optimizer_steps=steps))
        if device.type=='cuda':torch.cuda.synchronize()
        total=prev+time.perf_counter()-start
        if epoch==1 or epoch%args.val_every==0:
            base.emit(f'[EPOCH] {method} N{n}: {epoch}/{args.max_epochs} train={train_mse:.5g} '
                      f'valL2={val_full:.5g} best={best:.5g} at {best_epoch} '
                      f'time={total/60:.1f}min, steps={steps}')
        if epoch%args.checkpoint_every==0 or epoch==args.max_epochs or bad>=args.patience or \
              time.perf_counter()-args.session_start>=args.session_minutes*60:
            save_train_checkpoint(folder,model,opt,epoch,best,best_epoch,best_state,history,total,steps,cfg)
        if bad>=args.patience and epoch>=args.min_epochs:
            base.emit(f'[EARLY STOP] {method} N{n} epoch {epoch}');break
        if time.perf_counter()-args.session_start>=args.session_minutes*60:
            base.emit('[SESSION] Time budget exhausted; notebook can be rerun to resume')
            return None
    if best_state is None:raise RuntimeError(f'No best validation checkpoint for {method} N{n}')
    model.load_state_dict(best_state)
    pred,true=predict_complete(model,method,U,valids,device,mean,std,
                               1 if method=='CNO-DirectTime' else args.eval_batch)
    metrics=base.l2_metrics(pred,true)
    base.plot_error_curve(folder,metrics,'validation')
    base.plot_training(folder,history)
    base.plot_case(folder/'validation_vorticity_0.png',pred,true,0,f'{method} N{n} Validation case 200')
    duration=prev+time.perf_counter()-start
    torch.save(dict(state_dict=best_state,config=cfg,mean=mean,std=std),folder/'best_validation.pt')
    row=dict(method=method,N_train=n,config_hash=base.hash_config(cfg),version=VERSION,
       parameters=base.param_count(model),real_scalar_dof=base.real_dof(model),
       K=0 if method=='CNO-DirectTime' else 1,
       observed_input_frames=(10 if method=='UNO2D-H10' else 1),
       prediction_protocol=dict.fromkeys(['FFNO24-AR1','CNO-DirectTime','UNet-AR1','UNO2D-H10'],'' ).get(method,''),
       train_samples_per_epoch=len(dataset),optimizer_steps=steps,
       epochs_completed=history[-1]['epoch'],best_epoch=best_epoch,
       best_validation_full_l2=float(metrics['full'].mean()),
       best_validation_final_l2=float(metrics['final'].mean()),
       best_validation_forecast_from_t10_l2=float(forecast_from_t10_l2(pred,true).mean()),
       train_seconds=duration,checkpoint=str(folder/'best_validation.pt'),
       batch_size=args.batch_size,micro_batch_size=micro)
    row['prediction_protocol']={'FFNO24-AR1':'factorized_FFNO_24layer_autoregressive_one_step',
      'CNO-DirectTime':'direct_time_query_from_initial_only',
      'UNet-AR1':'autoregressive_single_step',
      'UNO2D-H10':'autoregressive_H10_40step_differentiable_training_10_true_test_initial_frames'}[method]
    base.atomic_json(done,row)
    del pred,true,model,opt
    if device.type=='cuda':torch.cuda.empty_cache()
    return row


def predeclared(args):
    return [dict(method=method,N_train=n,hash=base.hash_config(cfg_for(args,method,n)),
      path=str(folder_for(args,method,n))) for n in SIZES for method in METHODS]


def validation_table(args,plan):
    import pandas as pd
    rows=[json.loads((Path(j['path'])/'training_result.json').read_text())
          for j in plan if (Path(j['path'])/'training_result.json').is_file()]
    if rows:
        pd.DataFrame(rows).sort_values(['N_train','method']).to_csv(args.output/'validation_selection.csv',index=False)
        base.emit(f'[VALIDATION TABLE] {len(rows)}/{len(plan)} completed model fits')
    return rows


def evaluate_locked_test(U,args,plan,device):
    assert all((Path(j['path'])/'training_result.json').exists() for j in plan)
    previous=args.output/'LOCKED_TEST_PLAN.json'
    if previous.exists() and json.loads(previous.read_text())!=plan:
        raise ValueError('Locked test plan mismatch')
    base.atomic_json(previous,plan)
    rows=[]
    ids=np.arange(230,300)
    for j in plan:
        folder=Path(j['path']);done=folder/'test_result.json'
        if done.exists():rows.append(json.loads(done.read_text()));continue
        saved=torch.load(folder/'best_validation.pt',map_location='cpu',weights_only=False)
        assert base.hash_config(saved['config'])==j['hash']
        model=make_method(j['method']).to(device);model.load_state_dict(saved['state_dict'])
        model.eval();mean=saved['mean'];std=saved['std']
        if device.type=='cuda':torch.cuda.synchronize();torch.cuda.reset_peak_memory_stats()
        start=time.perf_counter()
        pred,true=predict_complete(model,j['method'],U,ids,device,mean,std,
                   1 if j['method']=='CNO-DirectTime' else args.eval_batch)
        infer_s=base.elapsed_s(device,start)
        peak=float(torch.cuda.max_memory_allocated(device)/2**20) if device.type=='cuda' else 0.0
        m=base.l2_metrics(pred,true);h1=physical_relative_h1(pred,true)
        bands=base.spectral_region_metrics(pred,true)
        tr=json.loads((folder/'training_result.json').read_text())
        row=dict(tr,full_l2_mean=float(m['full'].mean()),full_l2_std=float(m['full'].std()),
                 full_l2_median=float(np.median(m['full'])),
                 forecast_from_t10_l2_mean=float(forecast_from_t10_l2(pred,true).mean()),
                 forecast_from_t10_l2_std=float(forecast_from_t10_l2(pred,true).std()),final_l2_mean=float(m['final'].mean()),
                 final_l2_std=float(m['final'].std()),final_l2_median=float(np.median(m['final'])),
                 full_relative_h1_semi_mean=float(h1['full_relative_h1_semi'].mean()),
                 final_relative_h1_semi_mean=float(h1['final_relative_h1_semi'].mean()),
                 inference_seconds_70=infer_s,inference_seconds_per_trajectory=infer_s/70,
                 peak_inference_gpu_mib=peak,
                 **{key+'_'+stat:val for key,md in bands.items() for stat,val in md.items()})
        base.plot_error_curve(folder,m,'test')
        for i in (0,35):
            base.plot_case(folder/f'test_vorticity_{i}.png',pred,true,i,
                           f'{j["method"]} N{j["N_train"]} TEST case {230+i}')
        if j['N_train']==200:
            tframes=[0,25,50,75,100]
            np.savez_compressed(folder/'snapshot_frames_for_comparison.npz',
                frame_ids=np.array(tframes),case_ids=np.array([230,265]),
                truth=true[[0,35]][:,tframes],prediction=pred[[0,35]][:,tframes])
        base.atomic_json(done,row);rows.append(row)
        base.emit(f'[TEST] {j["method"]} N{j["N_train"]}: fullL2={row["full_l2_mean"]:.6g} '
                  f'finalL2={row["final_l2_mean"]:.6g} H1={row["full_relative_h1_semi_mean"]:.5g}')
        del pred,true,model
        if device.type=='cuda':torch.cuda.empty_cache()
        import pandas as pd
        pd.DataFrame(rows).sort_values(['N_train','method']).to_csv(args.output/'new_baselines_test_results.csv',index=False)
    return rows


def add_old_h1_and_images(U,args,device):
    # Original model files are only read after they have their own locked publication table.
    original=args.original_root/'publication_table.csv'
    if not original.is_file():return {}
    import pandas as pd
    df=pd.read_csv(original)
    if not {'AWFNO','TB-AWFNO','WFNO','WFNO-PM','FNO'}.issubset(set(df.method)):
        return {}
    info={}
    for _,r in df.loc[df.N_train==200].iterrows():
        ckpt=Path(r['checkpoint'])
        if not ckpt.is_file():continue
        s=torch.load(ckpt,map_location='cpu',weights_only=False)
        cfg=s['config']
        model=base.OperatorModel(cfg['method'],cfg['width'],cfg['modes'],cfg['depth'],
          cfg['k'],cfg['basis_rank'],cfg['order'],cfg['radius']).to(device)
        model.load_state_dict(s['state_dict']);model.eval()
        pred,true=base.model_predictions(model,U,[230,265],device,s['mean'],s['std'],1)
        grads=physical_relative_h1(pred,true)
        info[r['method']]={k+'_mean':float(v.mean()) for k,v in grads.items()}
        ids=[0,25,50,75,100]
        folder=args.output/'original_model_snapshots';folder.mkdir(exist_ok=True)
        np.savez_compressed(folder/f'{r["method"]}_N200.npz',frame_ids=np.array(ids),
                            case_ids=np.array([230,265]),truth=true[:,ids],prediction=pred[:,ids])
        del model,pred,true
    base.atomic_json(args.output/'original_model_h1_N200.json',info)
    return info


def visualize_all(args):
    import pandas as pd
    import matplotlib.pyplot as plt
    # Sources are strictly evaluated test rows from the independent locked plans.
    new=args.output/'new_baselines_test_results.csv'
    old=args.original_root/'publication_table.csv'
    if not new.is_file():return
    data=[pd.read_csv(new)]
    if old.is_file():data.insert(0,pd.read_csv(old))
    df=pd.concat(data,ignore_index=True,sort=False)
    df.sort_values(['N_train','method']).to_csv(args.output/'ALL_MODELS_COMPARISON.csv',index=False)
    if not old.is_file():
        base.emit('[MERGE] Original AWFNO/FNO locked test results not ready; new results saved separately.')
        return
    for key in ('full_l2_mean','final_l2_mean','train_seconds'):
        fig,ax=plt.subplots(figsize=(11,6),constrained_layout=True)
        for model in sorted(df.method.unique()):
            rr=df[df.method==model].sort_values('N_train')
            ax.plot(rr.N_train,rr[key],'-o',label=model)
        ax.set(xlabel='Training trajectories',ylabel=key,title=f'Navier-Stokes Re500: {key}')
        ax.legend(fontsize=8,ncol=2);ax.grid(alpha=.2)
        fig.savefig(args.output/f'ALL_MODELS_{key}.png',dpi=140);plt.close(fig)
    # Same underlying test example and identical truth/pred colour scale across models.
    sdir=args.output/'original_model_snapshots'
    source={}
    for f in sdir.glob('*_N200.npz'):
        source[f.stem.replace('_N200','')]=np.load(f)
    for method in METHODS:
        ff=folder_for(args,method,200)/'snapshot_frames_for_comparison.npz'
        if ff.is_file():source[method]=np.load(ff)
    if not source:return
    names=[q for q in ['AWFNO','TB-AWFNO','WFNO','WFNO-PM','FNO',*METHODS] if q in source]
    frames=[25,50,75,100];rowtruth=source[names[0]]['truth'][0]
    times=list(source[names[0]]['frame_ids'])
    truth=np.stack([rowtruth[times.index(t)] for t in frames])
    vm=float(np.max(np.abs(truth)))
    fig,axs=plt.subplots(1+len(names),len(frames),figsize=(14,2.5*(len(names)+1)),constrained_layout=True)
    for k,t in enumerate(frames):
        axs[0,k].imshow(truth[k].T,vmin=-vm,vmax=vm,origin='lower',cmap='RdBu_r')
        axs[0,k].set_title(f't={t*DT:.4f}')
    axs[0,0].set_ylabel('GROUND TRUTH',fontsize=9)
    for j,name in enumerate(names,1):
        arr=source[name];ids=list(arr['frame_ids'])
        for k,t in enumerate(frames):
            axs[j,k].imshow(arr['prediction'][0,ids.index(t)].T,vmin=-vm,vmax=vm,
                            origin='lower',cmap='RdBu_r')
        axs[j,0].set_ylabel(name,fontsize=9)
    for ax in axs.ravel():ax.set_xticks([]);ax.set_yticks([])
    fig.savefig(args.output/'ALL_MODELS_truth_predictions_N200_test230.png',dpi=145)
    plt.close(fig)
    for it in source.values():it.close()


def smoke(device):
    base.seed_all(1234)
    for method in METHODS:
        m=make_method(method).to(device)
        bs=1
        x=torch.randn(bs,2 if method=='CNO-DirectTime' else (10 if method=='UNO2D-H10' else 1),NX,NY,device=device)
        if method=='CNO-DirectTime':x[:,1]=0.35
        m.train();y=call_method(m,method,x)
        expected=1
        assert y.shape==(bs,expected,NX,NY),(method,y.shape)
        assert torch.isfinite(y).all(),method
        loss=y.square().mean();loss.backward()
        assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in m.parameters()),method
        base.emit(f'[SMOKE PASS] {method} output={tuple(y.shape)}, '
                  f'params={base.param_count(m):,} real_dof={base.real_dof(m):,}')
        del x,y,loss,m
        if device.type=='cuda':torch.cuda.empty_cache()
    err=np.zeros((1,NT,NX,NY),np.float32)
    exact=np.ones_like(err)
    h1=physical_relative_h1(err,exact)
    assert np.all(np.isfinite(h1['full_relative_h1_semi']))


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--dataset',type=Path,required=True)
    ap.add_argument('--output',type=Path,required=True)
    ap.add_argument('--original-root',type=Path,required=True)
    ap.add_argument('--max-epochs',type=int,default=45)
    ap.add_argument('--min-epochs',type=int,default=12)
    ap.add_argument('--patience',type=int,default=6)
    ap.add_argument('--val-every',type=int,default=3)
    ap.add_argument('--checkpoint-every',type=int,default=3)
    ap.add_argument('--batch-size',type=int,default=8)
    ap.add_argument('--micro-batch-cno',type=int,default=1)
    ap.add_argument('--micro-batch-other',type=int,default=4)
    ap.add_argument('--eval-batch',type=int,default=2)
    ap.add_argument('--seed',type=int,default=1234)
    ap.add_argument('--min-delta',type=float,default=1e-4)
    ap.add_argument('--lr',type=float,default=7e-4)
    ap.add_argument('--wd',type=float,default=1e-6)
    ap.add_argument('--session-minutes',type=float,default=180)
    ap.add_argument('--smoke-only',action='store_true')
    ap.add_argument('--test-only',action='store_true')
    ap.add_argument('--verify-file-hash',action='store_true')
    args=ap.parse_args()
    assert args.batch_size>0 and args.micro_batch_cno>0 and args.micro_batch_other>0
    assert args.min_epochs<=args.max_epochs and args.eval_batch>0
    args.output.mkdir(parents=True,exist_ok=True)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    args.session_start=time.perf_counter()
    base.emit(f'[CONFIG] {VERSION} torch={torch.__version__} device={device}')
    smoke(device)
    if args.smoke_only:return
    U=base.load_source(args.dataset,verify_hash=args.verify_file_hash)
    plan=predeclared(args)
    plan_path=args.output/'predeclared_extra_baselines.json'
    if plan_path.is_file():
        assert json.loads(plan_path.read_text())==plan,'Plan changed: use a new output root.'
    else:base.atomic_json(plan_path,plan)
    if not args.test_only:
        for i,j in enumerate(plan,1):
            if time.perf_counter()-args.session_start>args.session_minutes*60:break
            base.emit(f'[RUN {i}/{len(plan)}] {j["method"]} N{j["N_train"]}')
            out=fit_model(U,args,j['N_train'],j['method'],device)
            validation_table(args,plan)
            if out is None:break
    if all((Path(j['path'])/'training_result.json').is_file() for j in plan):
        base.emit('[TEST LOCK] All 16 new model fits completed and selected by validation.')
        evaluate_locked_test(U,args,plan,device)
        add_old_h1_and_images(U,args,device)
        visualize_all(args)
        base.emit('[FINISHED] Full model comparison tables and figures saved to '+str(args.output))
    else:
        base.emit('[SEALED TEST] New baseline matrix incomplete: rerun Colab to resume.')

if __name__=='__main__':main()


## 4. Readable checkpoint-reuse driver

This new driver only orchestrates cache discovery, completion of *non-F-FNO* fits and reporting. It creates a separate recovery output folder. Checkpoint keys and baseline source are unchanged.

In [ ]:
%%writefile /content/ns_recovery_driver.py
#!/usr/bin/env python3
"""Resume only CNO/U-Net/UNO using original Re500 v3 checkpoint keys.

No FFNO training. No architecture, training schedule, model code, or original
results are modified. This driver only creates supplementary summaries,
held-out evaluation artefacts and graphical diagnostics in a distinct folder.
"""
from __future__ import annotations
import argparse
import gc
import json
import math
import time
import traceback
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import ns_base_original as base
import ns_new_baselines as ns

METHODS = ('CNO-DirectTime', 'UNet-AR1', 'UNO2D-H10')
SIZES = tuple(base.SIZES)
FFNO = 'FFNO24-AR1'


def emit(msg):
    base.emit('[RECOVERY] '+msg)


def get_args(cli):
    # These values must remain equal to the source notebook v3: config hashes,
    # training folders and optimizer RNG checkpoints rely on them.
    return SimpleNamespace(
        output=cli.new_root, original_root=cli.original_root,
        max_epochs=45, min_epochs=12, patience=6, val_every=3,
        checkpoint_every=3, batch_size=8, micro_batch_cno=1,
        micro_batch_other=4, eval_batch=2, seed=1234,
        min_delta=1e-4, lr=7e-4, wd=1e-6,
        session_minutes=cli.session_minutes, session_start=time.perf_counter())


def training_folder(args,method,n):
    return ns.folder_for(args,method,n)


def file_from_run(args,method,n,basename):
    return training_folder(args,method,n)/basename


def training_rows(args, include_ffno=True):
    rows=[]
    methods=METHODS+((FFNO,) if include_ffno else ())
    for n in SIZES:
        for method in methods:
            f=file_from_run(args,method,n,'training_result.json')
            if f.is_file():
                row=json.loads(f.read_text())
                required=('method','N_train','config_hash','best_validation_full_l2','checkpoint')
                if any(k not in row for k in required):
                    raise RuntimeError(f'Incomplete result metadata: {f}')
                if row['method']!=method or int(row['N_train'])!=n:
                    raise RuntimeError(f'Corrupt/mismatched training record: {f}')
                expected=base.hash_config(ns.cfg_for(args,method,n))
                if row['config_hash']!=expected:
                    raise RuntimeError(f'Configuration mismatch: {f}')
                rows.append(dict(row))
    return rows


def print_matrix(df,col,label):
    print('\n'+'='*98,flush=True)
    print(label,flush=True)
    print('='*98,flush=True)
    if df.empty or col not in df.columns:
        print('None available yet',flush=True);return
    pivot=df.pivot_table(index='N_train',columns='method',values=col,aggfunc='first')
    pivot=pivot.reindex(index=SIZES)
    print(pivot.round(5).to_string(na_rep='--'),flush=True)


def write_validation(args,recovery):
    df=pd.DataFrame(training_rows(args))
    if not df.empty:
        df=df.sort_values(['N_train','method']).reset_index(drop=True)
        df.to_csv(recovery/'validation_all_cached_including_ffno.csv',index=False)
        (df.loc[df.method.isin(METHODS)]).to_csv(recovery/'validation_no_ffno.csv',index=False)
    else:
        # Keep an empty file for file browser discoverability.
        pd.DataFrame(columns=['method','N_train','best_validation_full_l2']).to_csv(
            recovery/'validation_no_ffno.csv',index=False)
    print_matrix(df,'best_validation_full_l2',
       'VALIDATION L2 | Existing completed checkpoints (cached F-FNO shown only, NEVER trained)')
    for n in SIZES:
        for method in METHODS:
            folder=training_folder(args,method,n)
            if (folder/'training_result.json').is_file():state='CACHED DONE'
            elif (folder/'checkpoint.pt').is_file():
                try:
                    ck=torch.load(folder/'checkpoint.pt',map_location='cpu',weights_only=False)
                    if ck['config']!=ns.cfg_for(args,method,n):
                        raise RuntimeError('CHECKPOINT CONFIG MISMATCH')
                    state=f'RESUMABLE epoch {ck["epoch"]}'
                except Exception as exc:
                    raise RuntimeError(f'Unsafe checkpoint: {folder}') from exc
            else:state='NOT STARTED'
            emit(f'{method:16s} N={n:3}  {state}')
    return df


def test_dir(recovery,method,n):
    return recovery/'test_runs'/method/f'N{n}'


def meanstd(a):
    a=np.asarray(a,dtype=np.float64)
    return (float(a.mean()),float(a.std(ddof=0)),float(np.median(a)))


def evaluate_one(U,args,recovery,device,method,n):
    output=test_dir(recovery,method,n)
    output.mkdir(parents=True,exist_ok=True)
    done=output/'test_result.json'
    if done.exists():
        emit(f'[CACHED TEST] {method} N{n} — no reevaluation')
        return json.loads(done.read_text())
    source=training_folder(args,method,n)
    record=source/'training_result.json'
    ck_path=source/'best_validation.pt'
    if not record.is_file() or not ck_path.is_file():
        emit(f'[SKIP TEST] {method} N{n}: no completed fit + best-validation weights')
        return None
    tr=json.loads(record.read_text())
    weights=torch.load(ck_path,map_location='cpu',weights_only=False)
    cfg=ns.cfg_for(args,method,n)
    assert tr['config_hash']==base.hash_config(cfg)
    assert weights['config']==cfg, f'Mismatched weights in {ck_path}'
    mean,std=weights['mean'],weights['std']
    model=ns.make_method(method).to(device)
    model.load_state_dict(weights['state_dict'])
    model.eval()
    # Evaluation is over the same held-out indices 230:300 for every method.
    # Stream cases in small batches to avoid materializing two 70x101x128x128 arrays.
    emit(f'[TEST START] {method} N{n}: 70 held-out trajectories, source={ck_path}')
    all_full=[];all_final=[];all_t10=[];all_h1=[];all_h1final=[];all_curve=[]
    final_p=[];final_t=[];case_snapshots={}
    if device.type=='cuda':
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)
    start=time.perf_counter()
    for start_idx in range(230,300,2):
        selected=list(range(start_idx,min(start_idx+2,300)))
        pred,true=ns.predict_complete(model,method,U,selected,device,mean,std,
                                      1 if method=='CNO-DirectTime' else 2)
        m=base.l2_metrics(pred,true)
        h1=ns.physical_relative_h1(pred,true)
        all_full.extend(m['full'].tolist());all_final.extend(m['final'].tolist())
        all_t10.extend(ns.forecast_from_t10_l2(pred,true).tolist())
        all_h1.extend(h1['full_relative_h1_semi'].tolist())
        all_h1final.extend(h1['final_relative_h1_semi'].tolist())
        all_curve.extend(m['per_timestep'].tolist())
        final_p.append(pred[:,-1].copy());final_t.append(true[:,-1].copy())
        for gid in (230,265):
            if gid in selected:
                idx=selected.index(gid)
                case_snapshots[gid]=(pred[idx:idx+1].copy(),true[idx:idx+1].copy())
        del pred,true,m,h1
        if (start_idx-230)%10==0:
            emit(f'[TEST PROGRESS] {method} N{n}: {min(start_idx-230+2,70)}/70')
    if device.type=='cuda':torch.cuda.synchronize(device)
    elapsed=time.perf_counter()-start
    peak=float(torch.cuda.max_memory_allocated(device)/2**20) if device.type=='cuda' else 0.
    # The reference spectral/region metric operates only on the final frame.
    fp=np.concatenate(final_p,axis=0)[:,None]
    ft=np.concatenate(final_t,axis=0)[:,None]
    spectral=base.spectral_region_metrics(fp,ft)
    full_mean,full_std,full_med=meanstd(all_full)
    final_mean,final_std,final_med=meanstd(all_final)
    t10_mean,t10_std,t10_med=meanstd(all_t10)
    row=dict(tr,full_l2_mean=full_mean,full_l2_std=full_std,full_l2_median=full_med,
             final_l2_mean=final_mean,final_l2_std=final_std,final_l2_median=final_med,
             forecast_from_t10_l2_mean=t10_mean,forecast_from_t10_l2_std=t10_std,
             forecast_from_t10_l2_median=t10_med,
             full_relative_h1_semi_mean=float(np.mean(all_h1)),
             final_relative_h1_semi_mean=float(np.mean(all_h1final)),
             inference_seconds_70=elapsed,
             inference_seconds_per_trajectory=elapsed/70,
             peak_inference_gpu_mib=peak,
             **{key+'_'+stat:val for key,v in spectral.items() for stat,val in v.items()})
    cases=pd.DataFrame(dict(test_trajectory=np.arange(230,300),full_l2=all_full,
          final_l2=all_final,forecast_from_t10_l2=all_t10,
          full_relative_h1_semi=all_h1,final_relative_h1_semi=all_h1final))
    cases.to_csv(output/'test_case_errors.csv',index=False)
    base.plot_error_curve(output,dict(per_timestep=np.asarray(all_curve)),'test')
    for gid,(pred,true) in case_snapshots.items():
        base.plot_case(output/f'test_vorticity_{gid}.png',pred,true,0,
                       f'{method} N{n}, TEST trajectory {gid}')
    if n==200:
        frames=np.asarray([0,25,50,75,100])
        ids=[gid for gid in (230,265) if gid in case_snapshots]
        np.savez_compressed(output/'snapshot_frames_for_comparison.npz',
                 frame_ids=frames,case_ids=np.asarray(ids),
                 truth=np.stack([case_snapshots[gid][1][0,frames] for gid in ids]),
                 prediction=np.stack([case_snapshots[gid][0][0,frames] for gid in ids]))
    base.atomic_json(output/'test_result.json',row)
    del model,weights,final_p,final_t,fp,ft
    gc.collect()
    if device.type=='cuda':torch.cuda.empty_cache()
    emit(f'[TEST DONE] {method} N{n} full={full_mean:.6f} final={final_mean:.6f} '
         f'forecast_t10={t10_mean:.6f}, elapsed={elapsed:.1f}s')
    return row


def evaluate_one_safely(U,args,recovery,device,method,n):
    """Preserve successful evaluations and show traceback if an individual run fails."""
    try:
        return evaluate_one(U,args,recovery,device,method,n)
    except Exception:
        message=traceback.format_exc()
        log=recovery/'evaluation_errors.log'
        with log.open('a') as out:
            out.write(f'\n===== {method} N={n} =====\n'+message+'\n')
        emit(f'[TEST FAILED] {method} N{n}. The traceback follows and is saved to {log}')
        print(message,flush=True)
        gc.collect()
        if device.type=='cuda':torch.cuda.empty_cache()
        return None


def test_rows(recovery):
    rows=[]
    for n in SIZES:
        for method in METHODS:
            f=test_dir(recovery,method,n)/'test_result.json'
            if f.is_file():rows.append(json.loads(f.read_text()))
    return pd.DataFrame(rows)


def summary_tables(args,recovery):
    from matplotlib import pyplot as plt
    fresh=test_rows(recovery)
    if fresh.empty:
        emit('No held-out metrics available yet; training/validation results saved.')
        return fresh
    fresh=fresh.sort_values(['N_train','method']).reset_index(drop=True)
    fresh.to_csv(recovery/'new_baselines_test_results.csv',index=False)
    original=args.original_root/'publication_table.csv'
    original_ready=original.is_file()
    if original_ready:
        old=pd.read_csv(original)
        assert {'N_train','method','full_l2_mean','final_l2_mean'}.issubset(old.columns)
        if old[['N_train','method']].duplicated().any():
            raise RuntimeError('Original publication table contains duplicate runs')
        old=old.copy()
        if 'observed_input_frames' not in old.columns:old['observed_input_frames']=1
        merged=pd.concat([old,fresh],ignore_index=True,sort=False)
        emit(f'ORIGINAL publication table found: {len(old)} rows, merged with {len(fresh)} new rows')
    else:
        merged=fresh.copy()
        emit('Original publication_table.csv absent: only new tested baselines displayed; no fabricated AWFNO scores.')
    merged=merged.sort_values(['N_train','method']).reset_index(drop=True)
    merged.to_csv(recovery/'ALL_MODELS_COMPARISON.csv',index=False)
    print_matrix(merged,'full_l2_mean',
        'HELD-OUT TEST | Physical full L2(Omega x T), 70 trajectories (UNO-H10 observes frames 0..9)')
    print_matrix(merged,'final_l2_mean',
        'HELD-OUT TEST | Final-time physical L2(Omega), 70 trajectories')
    print_matrix(merged,'forecast_from_t10_l2_mean',
        'HELD-OUT TEST | Physical L2(Omega x [t10,t100]) for new methods (ALL predicted for UNO)')
    # Boussinesq-style, sample-size curves: one figure for each common metric.
    for metric,label in [('full_l2_mean','Full-trajectory absolute physical L2'),
                         ('final_l2_mean','Final-time absolute physical L2'),
                         ('train_seconds','Total training seconds'),
                         ('full_relative_h1_semi_mean','Full-horizon relative H1 seminorm'),
                         ('final_relative_h1_semi_mean','Final-time relative H1 seminorm'),
                         ('high_frequency_relative_l2_mean','Final high-frequency relative L2'),
                         ('sharp_region_relative_l2_mean','Sharp-gradient-region relative L2'),
                         ('forecast_from_t10_l2_mean','Forecast [t10,t100] absolute physical L2')]:
        if metric not in merged.columns:
            emit(f'[PLOT SKIP] Missing metric column: {metric}')
            continue
        fig,ax=plt.subplots(figsize=(10,6),constrained_layout=True)
        for method in sorted(merged['method'].unique()):
            subset=merged[(merged.method==method)&merged[metric].notna()].sort_values('N_train')
            if subset.empty:continue
            ax.plot(subset.N_train,subset[metric],'-o',label=method)
        ax.set(xlabel='Training trajectories N',ylabel=label,title=f'Re=500 Kolmogorov flow: {label}')
        ax.grid(alpha=.25);ax.legend(fontsize=8,ncol=2)
        fig.savefig(recovery/('ALL_MODELS_'+metric+'.png'),dpi=150)
        plt.close(fig)
    # Common N=200 error-versus-time overlay; only saved/available model runs.
    fig,ax=plt.subplots(figsize=(10,5),constrained_layout=True)
    count=0
    for method in METHODS:
        file=test_dir(recovery,method,200)/'test_curves.npz'
        if file.is_file():
            with np.load(file) as curve:
                ax.plot(curve['t'],curve['mean'],label=method)
            count+=1
    if count:
        ax.set(xlabel='Forecast time since t=0.25',
               ylabel='Mean spatial absolute physical L2',
               title='Re=500: N200 held-out error versus forecast time')
        ax.legend();ax.grid(alpha=.25)
        fig.savefig(recovery/'N200_baselines_test_error_vs_time.png',dpi=150)
    plt.close(fig)
    return merged


def make_snapshots(U,args,recovery,device):
    """Comparable ground truth / prediction / error grids, N=200, cached after one creation."""
    from matplotlib import pyplot as plt
    f=recovery/'ALL_MODELS_truth_prediction_error_N200_test230.png'
    # Recompute on rerun if new model snapshots appear; N200 table is small.
    source={}
    for method in METHODS:
        archive=test_dir(recovery,method,200)/'snapshot_frames_for_comparison.npz'
        if archive.is_file():
            with np.load(archive) as a:
                source[method]=dict(truth=a['truth'][0],prediction=a['prediction'][0],
                                    frames=a['frame_ids'])
    original=args.original_root/'publication_table.csv'
    if original.is_file():
        old=pd.read_csv(original)
        for _,rec in old[old.N_train==200].iterrows():
            name=rec['method']
            archive=recovery/'original_snapshots'/f'{name}_N200.npz'
            if not archive.is_file() and Path(rec['checkpoint']).is_file():
                archive.parent.mkdir(parents=True,exist_ok=True)
                state=torch.load(rec['checkpoint'],map_location='cpu',weights_only=False)
                cfg=state['config']
                model=base.OperatorModel(cfg['method'],cfg['width'],cfg['modes'],cfg['depth'],
                    cfg['k'],cfg['basis_rank'],cfg['order'],cfg['radius']).to(device)
                model.load_state_dict(state['state_dict']);model.eval()
                pred,true=base.model_predictions(model,U,[230],device,state['mean'],state['std'],1)
                frames=np.asarray([0,25,50,75,100])
                np.savez_compressed(archive,frames=frames,truth=true[0,frames],prediction=pred[0,frames])
                del model,pred,true
                if device.type=='cuda':torch.cuda.empty_cache()
            if archive.is_file():
                with np.load(archive) as a:
                    source[name]=dict(truth=a['truth'],prediction=a['prediction'],frames=a['frames'])
    if not source:return
    order=[*base.MODELS,*METHODS]
    names=[s for s in order if s in source]
    frames=[25,50,75,100]
    truth_ref=source[names[0]]
    frame_map={int(x):i for i,x in enumerate(truth_ref['frames'])}
    truth=np.stack([truth_ref['truth'][frame_map[t]] for t in frames])
    vmax=max(float(np.abs(truth).max()),1e-8)
    # 3 rows per model: truth, prediction, absolute error.
    fig,axes=plt.subplots(3*len(names),len(frames),figsize=(14,3.9*len(names)),constrained_layout=True)
    axes=np.atleast_2d(axes)
    for i,name in enumerate(names):
        src=source[name]; fm={int(t):j for j,t in enumerate(src['frames'])}
        for j,t in enumerate(frames):
            gt=src['truth'][fm[t]];pred=src['prediction'][fm[t]]
            axs=axes[i*3:(i+1)*3,j]
            axs[0].imshow(gt.T,origin='lower',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
            axs[1].imshow(pred.T,origin='lower',cmap='RdBu_r',vmin=-vmax,vmax=vmax)
            err=np.abs(pred-gt)
            axs[2].imshow(err.T,origin='lower',cmap='magma',vmin=0,vmax=max(1e-8,float(np.quantile(err,.995))))
            if i==0:axs[0].set_title(f'Forecast t={t*base.DT:.4f}')
            for a in axs:a.set_xticks([]);a.set_yticks([])
        axes[3*i,0].set_ylabel(name+'  |  Truth',fontsize=9)
        axes[3*i+1,0].set_ylabel('Prediction',fontsize=9)
        axes[3*i+2,0].set_ylabel('|Error|',fontsize=9)
    fig.savefig(f,dpi=125)
    plt.close(fig)
    emit(f'[FIGURE] {f}')


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--dataset',type=Path,required=True)
    parser.add_argument('--original-root',type=Path,required=True)
    parser.add_argument('--new-root',type=Path,required=True)
    parser.add_argument('--session-minutes',type=float,default=180)
    parser.add_argument('--train-missing',action='store_true')
    parser.add_argument('--evaluate-completed',action='store_true')
    parser.add_argument('--only-summary',action='store_true')
    parser.add_argument('--skip-original-field-plots',action='store_true')
    args_cli=parser.parse_args()
    args=get_args(args_cli)
    if args_cli.train_missing and not torch.cuda.is_available():
        raise RuntimeError('GPU required to resume training')
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    recovery=args.output/'RECOVERY_COMPARISON_NO_FFNO_v1'
    recovery.mkdir(parents=True,exist_ok=True)
    emit(f'Existing checkpoints: {args.output}')
    emit(f'NEW results, without overwriting original files: {recovery}')
    emit('F-FNO remains implemented and its N25/N50 checkpoints are preserved; NO F-FNO FIT OR TEST here.')
    write_validation(args,recovery)
    summary_tables(args,recovery)
    if args_cli.only_summary:return
    U=base.load_source(args_cli.dataset,verify_hash=False)
    if args_cli.evaluate_completed:
        # Immediate recovery of already completed work; no training required.
        for n in SIZES:
            for method in METHODS:
                if file_from_run(args,method,n,'training_result.json').is_file():
                    evaluate_one_safely(U,args,recovery,device,method,n)
                    summary_tables(args,recovery)
        if not args_cli.skip_original_field_plots:
            try: make_snapshots(U,args,recovery,device)
            except Exception:
                message=traceback.format_exc()
                (recovery/'original_field_plot_errors.log').write_text(message)
                emit('[ORIGINAL PLOT ERROR] Existing test results remain saved')
                print(message,flush=True)
    if args_cli.train_missing:
        for n in SIZES:
            for method in METHODS:
                if time.perf_counter()-args.session_start>=args.session_minutes*60:
                    emit('TIME BUDGET REACHED. Checkpoints preserved. Rerun same notebook to resume.')
                    write_validation(args,recovery)
                    summary_tables(args,recovery)
                    return
                f=file_from_run(args,method,n,'training_result.json')
                if f.is_file():
                    emit(f'[SKIP TRAINING; CACHED] {method} N{n}')
                    continue
                emit(f'[FIT / RESUME] {method} N{n}, old checkpoint automatically reused')
                result=ns.fit_model(U,args,n,method,device)
                write_validation(args,recovery)
                if result is None:
                    emit('PARTIAL CHECKPOINT SAVED. Rerun this notebook to continue from its last epoch.')
                    return
                if args_cli.evaluate_completed:
                    evaluate_one_safely(U,args,recovery,device,method,n)
                    summary_tables(args,recovery)
                    if n==200 and not args_cli.skip_original_field_plots:
                        try: make_snapshots(U,args,recovery,device)
                        except Exception:
                            message=traceback.format_exc()
                            (recovery/'original_field_plot_errors.log').write_text(message)
                            emit('[ORIGINAL PLOT ERROR] Existing test results remain saved')
                            print(message,flush=True)
    write_validation(args,recovery)
    summary_tables(args,recovery)
    if not args_cli.skip_original_field_plots:
        try: make_snapshots(U,args,recovery,device)
        except Exception:
            message=traceback.format_exc()
            (recovery/'original_field_plot_errors.log').write_text(message)
            emit('[ORIGINAL PLOT ERROR] Existing test results remain saved')
            print(message,flush=True)
    completed=sum(file_from_run(args,m,n,'training_result.json').is_file()
                  for n in SIZES for m in METHODS)
    evaluated=len(test_rows(recovery))
    emit(f'COMPLETED {completed}/12 non-F-FNO fits; TEST RESULTS {evaluated}/12. '
         f'Resume same notebook if less than 12. Files: {recovery}')


if __name__=='__main__':main()


In [ ]:
#@title 5. Verify source and print the complete cached-run status NOW
for file in ['/content/ns_base_original.py','/content/ns_new_baselines.py','/content/ns_recovery_driver.py']:
    py_compile.compile(file,doraise=True)
print('All three source files compiled successfully.',flush=True)
cmd=[sys.executable,'-u','/content/ns_recovery_driver.py',
     '--dataset',str(DATASET),'--original-root',str(ORIGINAL_ROOT),
     '--new-root',str(NEW_ROOT),'--only-summary']
subprocess.run(cmd,check=True)


In [ ]:
#@title 6. FIRST: Recover held-out test results from completed CNO/U-Net/UNO checkpoints (NO training)
if EVALUATE_FINISHED:
    cmd=[sys.executable,'-u','/content/ns_recovery_driver.py',
         '--dataset',str(DATASET),'--original-root',str(ORIGINAL_ROOT),
         '--new-root',str(NEW_ROOT),'--evaluate-completed',
         '--session-minutes',str(SESSION_MINUTES)]
    if not PLOT_ORIGINAL_FIELDS:cmd+=['--skip-original-field-plots']
    print('Evaluating ONLY completed, validation-selected saved checkpoints. No retraining.',flush=True)
    log=RECOVERY_ROOT/'evaluation_stdout.log'
    print('Full subprocess output:',log,flush=True)
    with log.open('a',buffering=1) as stream:
        process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                 text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            stream.write(line)
        rc=process.wait()
    if rc:
        raise RuntimeError(f'Python evaluation exited with status {rc}. Full traceback: {log}')
else:
    print('Existing validation results are still accessible; held-out test step disabled.',flush=True)


## 7. Resume incomplete non-F-FNO runs

Existing complete fits print **CACHED**; partially trained checkpoints print **RESUME** with their saved epoch; only completely missing runs start at epoch 1. Each completed fit updates the validation matrix and (when enabled) the held-out test CSV/figures. Training prints epoch progress. Colab limits can stop the notebook mid-study: rerun **Run all** to continue.

In [ ]:
#@title 7. Resume remaining CNO/U-Net/UNO training, print cumulative comparisons, and checkpoint automatically
from threading import Thread, Event
import time, datetime
if TRAIN_MISSING:
    cmd=[sys.executable,'-u','/content/ns_recovery_driver.py',
         '--dataset',str(DATASET),'--original-root',str(ORIGINAL_ROOT),
         '--new-root',str(NEW_ROOT),'--train-missing',
         '--session-minutes',str(SESSION_MINUTES)]
    if EVALUATE_FINISHED:cmd+=['--evaluate-completed']
    if not PLOT_ORIGINAL_FIELDS:cmd+=['--skip-original-field-plots']
    log=RECOVERY_ROOT/'recovery_colab_stdout.log'
    print('LIVE LOG:',log,flush=True)
    print('The previously saved FFNO fits are NOT included in this training plan.',flush=True)
    last_seen=[time.monotonic()]
    stop_heartbeat=Event()
    def heartbeat():
        while not stop_heartbeat.wait(45):
            print(f'[{datetime.datetime.now():%H:%M:%S}] Colab still running;'
                  f' latest model log {int(time.monotonic()-last_seen[0])}s ago.',flush=True)
    thread=Thread(target=heartbeat,daemon=True);thread.start()
    try:
        with open(log,'a',buffering=1) as stream:
            proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                                  text=True,bufsize=1)
            for line in proc.stdout:
                last_seen[0]=time.monotonic()
                print(line,end='',flush=True)
                stream.write(line)
            rc=proc.wait()
            if rc!=0:raise RuntimeError(f'Recovery process exited with code {rc}; see {log}')
    finally:
        stop_heartbeat.set();thread.join(timeout=1)
else:
    print('Training disabled. All cached checkpoints remain unchanged.',flush=True)


In [ ]:
#@title 8. Boussinesq-style tables + view saved plots (can rerun anytime)
from pathlib import Path
import pandas as pd
from IPython.display import display,Image
files=['validation_all_cached_including_ffno.csv','validation_no_ffno.csv',
       'new_baselines_test_results.csv','ALL_MODELS_COMPARISON.csv']
for name in files:
    file=RECOVERY_ROOT/name
    print(name,'EXISTS' if file.is_file() else 'not produced yet',flush=True)
    if file.is_file():
        df=pd.read_csv(file)
        keys=[c for c in ['N_train','method','observed_input_frames','parameters',
                 'real_scalar_dof','best_epoch','best_validation_full_l2','full_l2_mean',
                 'forecast_from_t10_l2_mean','final_l2_mean','full_relative_h1_semi_mean',
                 'train_seconds','inference_seconds_70'] if c in df.columns]
        display(df[keys].sort_values(['N_train','method']) if keys else df)
for filename in ['ALL_MODELS_full_l2_mean.png','ALL_MODELS_final_l2_mean.png',
     'ALL_MODELS_forecast_from_t10_l2_mean.png',
     'ALL_MODELS_truth_prediction_error_N200_test230.png']:
    f=RECOVERY_ROOT/filename
    if f.exists():
        print('Figure:',f.name)
        display(Image(filename=str(f)))
print('Permanent results:',RECOVERY_ROOT)
print('Continue later: Runtime → Run all (checkpoints are reused).')
